# Grounded Investment Assistant — Full Code Notebook v3

This version preserves the complete Full Code v2 RAG workflow while using small, single-purpose cells inspired by Low Code v2. Every code cell states its objective, and every function includes a concise docstring.

**Rubric path:** data preparation (1.1–1.3) → grounded question answering (1.4) → evaluation (1.5–1.7) → GEPA prompt optimization (2) → complete RAG configuration tuning (3) → five official test cases (4).


## Problem Statement

### Business Context

GlobalEdge Brokerage is a mid-sized brokerage firm operating across multiple countries, serving thousands of retail and institutional clients through a network of equity brokers. Brokers handle investment recommendations, portfolio reviews, and market advisory discussions across equity, forex, commodity, crypto, and international stock markets. Each broker manages hundreds of clients and is expected to stay updated with overnight market developments before the market opens.

Every morning, brokers must review large volumes of financial news, stock-price movements, earnings discussions, analyst commentary, and SEC regulatory filings within a very limited preparation window. In practice, brokers can only read a small portion of the available information before their first client calls begin. As a result, many important signals, disclosures, and market events remain unnoticed.

This creates an intelligence gap during client interactions. Brokers often rely on partial information, memory, or fragmented market sources while answering client questions. Important disclosures buried inside long 10-K and 10-Q filings are difficult to review manually, and brokers may struggle to provide evidence-backed responses when clients ask for justification or supporting references. This increases both compliance risk and trust-related challenges.

To solve this problem, GlobalEdge wants to build a financial intelligence assistant that allows brokers to ask natural-language questions and receive grounded answers backed by real financial data, news articles, and regulatory filings. The system should reduce information overload, improve market coverage, and help brokers make faster and more confident advisory decisions without adding technical complexity to their workflow.

### Objective

This project proposes a proof-of-concept Retrieval-Augmented Generation (RAG) financial intelligence system that combines financial news, stock-price data, and SEC filings into a searchable intelligence layer. The system uses semantic retrieval with a local Chroma vector database to fetch relevant financial context and generate grounded answers for broker questions using large language models. Brokers can ask plain-English questions such as market sentiment analysis, company-risk queries, filing-related questions, or cross-market comparisons and receive evidence-backed responses.

To improve the quality and reliability of generated answers, the system introduces a DeepEval-based prompt optimization workflow using GEPA (Genetic-Pareto Prompt Optimization). Instead of manually refining prompts, the workflow uses benchmark financial question-answer examples and evaluation metrics such as faithfulness, groundedness, relevance, and actionability to optimize the final answering prompt.

The project also evaluates multiple RAG configurations by tuning retrieval and generation parameters such as chunk size, chunk overlap, number of retrieved chunks, temperature, top-p, and maximum token limits. Each configuration is evaluated using DeepEval metrics to identify the most effective combination for grounded financial reasoning and broker-style question answering.

The final system uses the optimized prompt together with the best-performing RAG configuration to generate accurate, grounded, and production-style financial intelligence responses for broker workflows.

### Data Description

The system uses three financial intelligence datasets collected through a custom data ingestion pipeline and stored locally for the RAG workflow.
1. Global Financial News (global_news.csv)
Contains financial news articles with titles, article content, publication dates, URLs, and source metadata. The dataset covers company announcements, market developments, macroeconomic events, sector movements, and sentiment-related signals.
2. Global Stock Prices (all_prices_clean.csv)
Contains historical stock-price data across global equity markets, crypto markets, and forex markets. Each record includes ticker symbols, company names, open/high/low/close prices, trading volume, and timestamps for market analysis and trend evaluation.
3. SEC Regulatory Filings (sec_filings.txt)
Contains regulatory filing documents including 10-K annual reports, 10-Q quarterly reports, and other SEC disclosures. These filings include financial statements, governance information, risk disclosures, operational commentary, and compliance-related information used for grounded financial reasoning.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Cell objective: Install pinned dependencies required by the complete notebook.
# Objective: install the compatible libraries required through Section 2.4.
# Run this once, restart the kernel as noted below, and then execute sequentially.
%pip install -qU \
    "chromadb==1.5.9" \
    "langchain-community==0.4.1" \
    "langchain-chroma==1.1.0" \
    "langchain-openai==1.2.1" \
    "langchain-text-splitters==1.1.2" \
    "pandas>=2.2.0" \
    "numpy>=1.26.0" \
    "pypdf>=5.0.0" \
    "scikit-learn>=1.4.0" \
    "python-dotenv>=1.0.1" \
    "tqdm>=4.66.0" \
    "deepeval==4.0.9"


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# Cell objective: Import standard Python and tabular-data utilities.
# Importing the necessary libraries
# Objective: centralize every dependency used through Section 2.4 so that
# the notebook execution order and external requirements remain explicit.

# Standard Python libraries for file handling, text processing, JSON handling,
# randomness, timing, and basic utilities.
import os
import re
import json
import random
import time
from pathlib import Path
from collections import Counter
from hashlib import sha256
import pandas as pd
from dotenv import load_dotenv


In [2]:
# Cell objective: Import DeepEval components for evaluation and GEPA optimization.
# DeepEval libraries for prompt optimization and evaluation.
import deepeval
from deepeval.prompt import Prompt
from deepeval.dataset import Golden
from deepeval.metrics import GEval
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import GPTModel
from deepeval.optimizer.policies import TieBreaker


In [3]:
# Cell objective: Import LangChain components for documents, retrieval, and generation.
# LangChain libraries for document loading, text splitting, embeddings,
# vector storage, and LLM interaction.
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document


## Environment Setup and Project Initialization

In [4]:
# Cell objective: Load the project environment and fail clearly when the API key is missing.
env_path = Path("../.env") if Path("../.env").exists() else Path(".env")
load_dotenv(env_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found. Add it to your .env file before running the notebook.")

print(f"Loaded environment variables from: {env_path.resolve()}")


Loaded environment variables from: /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/.env


# Section 1: Baseline Retrieval-Augmented Generation (RAG) Pipeline

## 1.1 Load the CSV files with `CSVLoader`

In [5]:
# Cell objective: Resolve and load the two CSV inputs without changing raw data.
# Objective: build explicit LangChain Documents without changing the raw CSVs.
# page_content is optimized for embeddings, while metadata supports filtering,
# traceability, citations, and deterministic retrieval diagnostics.

# Resolve the raw-data directory for both common execution locations:
# running from the notebooks folder or from the project root.
data_dir = Path("../data/raw") if Path("../data/raw").exists() else Path("data/raw")

# Load both CSV sources into DataFrames. Missing values are replaced with
# empty strings so document text and Chroma metadata remain serializable.
stock_prices_path = data_dir / "stock_price_details.csv"
global_news_path = data_dir / "global_news.csv"
stock_df = pd.read_csv(stock_prices_path).fillna("")
news_df = pd.read_csv(global_news_path).fillna("")


In [6]:
# Cell objective: Define how structured stock identifiers are parsed from price summaries.
# Parse the structured identifiers embedded in each human-readable price
# summary. These fields become metadata filters without modifying the CSV.
price_record_pattern = re.compile(
    r"Date is (?P<date>[^,]+), ticker is (?P<ticker>[^,]+), "
    r"name is (?P<name>[^,]+),"
)


In [7]:
# Cell objective: Convert validated stock rows into traceable LangChain Documents.
# Convert every valid stock-price row into one LangChain Document. The full
# summary is embedded, while ticker and date support exact metadata filtering.
stock_docs = []
unparsed_price_rows = []
for row_id, row in stock_df.iterrows():
    price_summary = str(row["price_summary"]).strip()

    match = price_record_pattern.search(price_summary)

    if not match:
        unparsed_price_rows.append(row_id)
        continue

    fields = match.groupdict()

    stock_docs.append(Document(
        page_content=price_summary,                     # the full price summary is embedded for semantic search. Chroma generates the embeddings from this field.  
        metadata={                                      # metadata fields are used for filtering, traceability, and provenance. They are not embedded.
            "dataset": "stock_price_details",           # chroma metadata filter for the source dataset:
            "source_file": stock_prices_path.name,          # chroma metadata filter for the source CSV
            "row_id": int(row_id),                          # chroma metadata filter for the source CSV row   
            "ticker": fields["ticker"].strip().upper(),     # chroma metadata filter for the stock ticker
            "date": fields["date"].strip(),                 # chroma metadata filter for the stock price date
            "name": fields["name"].strip(),                 # chroma metadata filter for the stock name
        },
    ))

# Fail early instead of silently indexing incomplete price records. This keeps
# the indexed document count aligned with the source dataset.
if unparsed_price_rows:
    raise ValueError(f"Could not parse stock rows: {unparsed_price_rows[:10]}")


In [10]:
# Cell objective: Convert news rows into semantic text plus filterable provenance metadata.
# Build one news Document per CSV row. The title, date, source, description,
# and content form the semantic text; provenance fields remain in metadata.
news_docs = []
for row_id, row in news_df.iterrows():
    news_content = f"""
Title: {row['title']}
Published at: {row['published_at']}
Source: {row['source']}

Description:
{row['description']}

Content:
{row['content']}
""".strip()

    news_docs.append(Document(
        page_content=news_content,                      # the full news text is embedded for semantic search. Chroma generates the embeddings from this field.
        metadata={                                      # metadata fields are used for filtering, traceability, and provenance. They are not embedded.    
            "dataset": "global_news",                       # chroma metadata filter for the source dataset
            "source_file": global_news_path.name,           # chroma metadata filter for the source CSV
            "row_id": int(row_id),                          # chroma metadata filter for the source CSV row    
            "title": str(row["title"]),                     # chroma metadata filter for the news title
            "source": str(row["source"]),                   # chroma metadata filter for the news source
            "url": str(row["url"]),                         # chroma metadata filter for the news URL
            "published_at": str(row["published_at"]),       # chroma metadata filter for the news publication date
            "query": str(row["query"]),                     # chroma metadata filter for the news query
        },
    ))


In [11]:
# Cell objective: Combine CSV documents and display a compact ingestion check.
# Combine the two CSV document collections for downstream indexing and print
# a compact ingestion check before the SEC documents are added in Section 1.2.
csv_docs = stock_docs + news_docs

print(f"Loaded {len(stock_docs)} stock price documents")
print(f"Loaded {len(news_docs)} news documents")
print(f"Loaded {len(csv_docs)} total CSV documents")
print("Sample metadata:", csv_docs[0].metadata)
print(csv_docs[0].page_content[:500])


Loaded 500 stock price documents
Loaded 500 news documents
Loaded 1000 total CSV documents
Sample metadata: {'dataset': 'stock_price_details', 'source_file': 'stock_price_details.csv', 'row_id': 0, 'ticker': '0700.HK', 'date': '2026-03-12', 'name': 'Tencent (Hong Kong)'}
On this record, Date is 2026-03-12, ticker is 0700.HK, name is Tencent (Hong Kong), Open is 550.5, High is 557.0, Low is 541.5, Close is 546.5, Volume is 19663840.0.


## 1.2 Load the SEC filings text and split it into chunks


In [12]:
# Cell objective: Resolve and validate the SEC filing input path.
# Objective: preserve page-level provenance while creating embedding-sized
# SEC chunks. The PDF remains unchanged; only in-memory metadata is enriched.
sec_pdf_path = data_dir / "sec_filings_10q.pdf"

if not sec_pdf_path.exists():
    raise FileNotFoundError("Expected sec_filings_10q.pdf in the raw data folder.")


In [13]:
# Cell objective: Load the PDF by page and enrich page-level provenance metadata.
# Load the SEC PDF into one LangChain Document per page. The PyPDFLoader
# automatically extracts text and preserves page-level metadata. 
sec_loader = PyPDFLoader(str(sec_pdf_path))

# sec_raw_docs is a list of Document objects, each representing a page in the SEC PDF. 
# Each Document object contains the text content of the page and metadata such as the page number and source file name.
sec_raw_docs = sec_loader.load()

# Fail early if the SEC PDF did not produce any page documents. This ensures that the subsequent processing steps have valid input data to work with.
if not sec_raw_docs:
    raise ValueError("The SEC PDF did not produce any page documents.")


# Enrich the SEC page-level metadata with dataset, source file, and document type.
for doc in sec_raw_docs:
    doc.metadata.update({
        "dataset": "sec_filings",
        "source_file": sec_pdf_path.name,
        "document_type": "10-Q",
    })


In [14]:
# Cell objective: Split SEC pages into overlapping chunks and assign traceable chunk IDs.
# Objective: create embedding-sized chunks from the SEC filings while preserving
# page-level provenance. The RecursiveCharacterTextSplitter is configured to
# produce chunks of 1000 characters with 200 characters of overlap. This ensures
# that each chunk is large enough for semantic search while maintaining context
# across chunk boundaries. The overlap helps to preserve continuity in the text,
# which is important for understanding the content in a financial document like a 10-Q filing.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

# Split the SEC documents into embedding-sized chunks.
sec_docs = text_splitter.split_documents(sec_raw_docs)

# Add a unique chunk ID to each SEC chunk for tracking and retrieval.
for chunk_id, doc in enumerate(sec_docs):
    doc.metadata["chunk_id"] = chunk_id

# Fail early if the SEC chunking produced empty content. This ensures that the subsequent processing steps have valid input data to work with.
if not sec_docs or any(not doc.page_content.strip() for doc in sec_docs):
    raise ValueError("SEC chunking produced empty content.")


In [15]:
# Cell objective: Combine CSV documents and SEC chunks for vector indexing.
# Combine the CSV documents with the SEC chunks for final indexing and print a compact ingestion check. 
# This ensures that all documents are ready for vector indexing and semantic search.
all_docs = csv_docs + sec_docs

print(f"Loaded {len(sec_raw_docs)} SEC filing source documents")
print(f"Created {len(sec_docs)} SEC filing chunks")
print(f"Prepared {len(all_docs)} total documents for vector indexing")
print(sec_docs[0].page_content[:500])


Loaded 126 SEC filing source documents
Created 522 SEC filing chunks
Prepared 1522 total documents for vector indexing
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
________________________________________________________________________________________
FORM 10-Q
_______________________________________________________________________________________
(Mark One)
☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the quarterly period ended March 31, 2026OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
F


## 1.3 Build a local Chroma vector store

In [16]:
# Cell objective: Configure the embedding model and persistent Chroma collection.
# Objective: rebuild a deterministic Version 2 collection on every run.
# This prevents duplicate documents from changing top-k results after a cell rerun.
#
# Model rationale: text-embedding-3-small was selected as an efficient and
# cost-effective embedding model with strong semantic retrieval capabilities.
# It converts news articles, stock-price records, and SEC filing chunks into
# comparable vectors in one Chroma collection. Its quality is sufficient for
# this proof-of-concept corpus while keeping repeated indexing and configuration
# experiments fast, affordable, and reproducible. Although
# text-embedding-3-large may improve retrieval in more demanding production
# workloads, the small model provides the appropriate balance of retrieval
# performance, speed, storage efficiency, and cost for this project.
chroma_dir = Path("../chroma_db") if Path("../data/raw").exists() else Path("chroma_db")
collection_name = "investment_rag_v2"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


In [17]:
# Cell objective: Reset the collection so notebook reruns remain reproducible.
# Create a new Chroma client and delete the existing collection if it exists
chroma_client = chromadb.PersistentClient(path=str(chroma_dir))
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    # A missing collection is expected on the first run.
    pass


In [19]:
# Cell objective: Generate deterministic document IDs and reject duplicate identities.
# Define a function to generate deterministic document IDs based on the document's metadata and content. 
# This ensures that each document has a unique identifier that can be used for indexing and retrieval.
def deterministic_document_id(doc):
    """Return a reproducible SHA-256 ID from a document's metadata and content."""
    identity = json.dumps(doc.metadata, sort_keys=True) + doc.page_content
    return sha256(identity.encode("utf-8")).hexdigest()

# Generate deterministic document IDs for all documents and check for duplicates before indexing. 
# This ensures that each document can be uniquely identified and retrieved later.
document_ids = [deterministic_document_id(doc) for doc in all_docs]
if len(document_ids) != len(set(document_ids)):
    raise ValueError("Duplicate document identities were detected before indexing.")


In [20]:
# Cell objective: Index all documents in one Chroma collection.
# Create a new Chroma collection with the specified name, embedding function, and client.
vector_store = Chroma(
    client=chroma_client,
    collection_name=collection_name,
    embedding_function=embeddings,
)

# Add all documents to the Chroma collection with their corresponding deterministic IDs. 
# This ensures that each document is indexed and can be retrieved later using its unique identifier.
vector_store.add_documents(all_docs, ids=document_ids)


['...',
 ...]

In [21]:
# Cell objective: Expose the baseline retriever and verify indexed document counts.
# Create a retriever from the vector store with similarity search and a top-k of 5.
# This allows for efficient retrieval of the most relevant documents based on their embeddings when a query is made.
# search_type="similarity" specifies that the retriever will use cosine similarity to find the most relevant documents based on their embeddings.
# search_kwargs={"k": 5} specifies that the retriever will return the top 5 most similar documents for a given query.
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

dataset_counts = Counter(doc.metadata["dataset"] for doc in all_docs)
print(f"Created Chroma collection: {collection_name}")
print(f"Persisted vector store at: {chroma_dir.resolve()}")
print(f"Indexed {vector_store._collection.count()} documents")
print("Documents by dataset:", dict(dataset_counts))


Created Chroma collection: investment_rag_v2
Persisted vector store at: /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/chroma_db
Indexed 1522 documents
Documents by dataset: {'stock_price_details': 500, 'global_news': 500, 'sec_filings': 522}


## 1.4 Retrieve Context and Generate Answers using the Baseline RAG Pipeline

In [22]:
# Cell objective: Define the grounded-answer policy and baseline system prompt.
# Objective: make retrieval observable and source-aware while preserving a
# transparent similarity-search baseline. Routing uses only the user question.
FINANCIAL_SAFETY_POLICY = """
Do not provide personalized buy, sell, or hold recommendations. Present the
available evidence, risks, signals, and data limitations so the broker or client
can make an informed decision.
""".strip()

baseline_system_prompt = f"""
You are a grounded investment intelligence assistant for brokerage analysts.
Use only the retrieved context to answer the user's question.
If the context is insufficient, say what is missing instead of guessing.
Keep the answer concise, evidence-backed, and suitable for a broker speaking with a client.
{FINANCIAL_SAFETY_POLICY}
""".strip()


In [23]:
# Cell objective: Define helpers that turn retrieved documents into cited model context.
# Define a function to format source metadata for display. 
def format_source_metadata(metadata):
    """Convert document metadata into a compact, human-readable source label."""
    source_file = metadata.get("source_file") or metadata.get("source") or metadata.get("dataset", "unknown")
    title = metadata.get("title")
    published_at = metadata.get("published_at")
    page = metadata.get("page")
    row = metadata.get("row_id", metadata.get("row"))

    details = [str(source_file)]
    if title:
        details.append(str(title))
    if published_at:
        details.append(str(published_at))
    if page is not None:
        details.append(f"page {page}")
    if row is not None:
        details.append(f"row {row}")

    return " | ".join(details)

# Define a function to format the retrieved context for display.
def format_retrieved_context(docs):
    """Format retrieved documents as numbered evidence for the answer model."""
    formatted_sources = []

    for idx, doc in enumerate(docs, start=1):
        metadata = doc.metadata or {}
        source_label = format_source_metadata(metadata)
        content = re.sub(r"\s+", " ", doc.page_content).strip()
        formatted_sources.append(f"[Source {idx}] {content}\nMetadata: {source_label}")

    source_text = "\n\n".join(formatted_sources)
    return f"Corpus snapshot date: {CORPUS_AS_OF_DATE}\n\n{source_text}"


In [24]:
# Cell objective: Define source-routing vocabulary for prices, filings, and news.
# Define sets of terms to help route questions to the appropriate datasets. 
PRICE_TERMS = {
    "price", "close", "closing", "open", "high", "low",
    "volume", "trading volume", "exchange rate", "stock",
    "price trend", "performing",
}
SEC_TERMS = {
    "10-k", "10-q", "sec", "filing", "auditor", "audit",
    "internal control", "material weakness", "financial statements",
    "financials", "revenue", "net income", "risk", "risks",
}
NEWS_TERMS = {
    "news", "recent reports", "market chatter", "announced",
    "outlook", "blockade", "recently", "geopolitical",
    "exchange and ticker", "sentiment", "buzz", "stepping down",
}


In [25]:
# Cell objective: Build company aliases and known tickers from ingested stock metadata.
# Build aliases from stock metadata and add only the common-name exceptions
# that cannot be derived reliably from the source labels.
COMPANY_TICKER_ALIASES = {}
for doc in stock_docs:
    company_name = str(doc.metadata["name"]).strip().lower()
    base_name = re.sub(r"\s*\([^)]*\)\s*$", "", company_name).strip()
    COMPANY_TICKER_ALIASES[company_name] = doc.metadata["ticker"]
    COMPANY_TICKER_ALIASES[base_name] = doc.metadata["ticker"]

COMPANY_TICKER_ALIASES.update({
    "apple": "AAPL",
    "amazon": "AMZN",
    "alphabet": "GOOGL",
    "google": "GOOGL",
    "tesla": "TSLA",
    "tencent": "0700.HK",
    "bitcoin": "BTC-USD",
    "usd/jpy": "USDJPY=X",
    "usdjpy": "USDJPY=X",
})

# Build a sorted list of known tickers from the stock documents for efficient extraction.
KNOWN_TICKERS = sorted({doc.metadata["ticker"] for doc in stock_docs}, key=len, reverse=True)


In [33]:
# Cell objective: Derive the corpus cutoff date from the available source records.
# Derive the corpus cutoff from the available news and price records instead
# of hard-coding a date that could become stale when the raw files change.
news_dates = pd.to_datetime(news_df["published_at"], errors="coerce", utc=True)
price_dates = pd.to_datetime(
    [doc.metadata["date"] for doc in stock_docs], errors="coerce", utc=True
)
available_dates = pd.concat([
    pd.Series(news_dates).dropna(),
    pd.Series(price_dates).dropna(),
])
if available_dates.empty:
    raise ValueError("Could not derive the corpus snapshot date.")
CORPUS_AS_OF_DATE = available_dates.max().date().isoformat()


In [34]:
# Cell objective: Define the retrieval plan and metadata filters derived from a question.
# Define a function to route questions to the appropriate datasets based on keywords in the question.
def route_question(question):
    """Select the datasets that are most likely to answer the question."""
    text = question.lower()
    constraints = extract_question_constraints(question)
    datasets = []
    if any(term in text for term in PRICE_TERMS):
        datasets.append("stock_price_details")
    if any(term in text for term in SEC_TERMS):
        datasets.append("sec_filings")
    if any(term in text for term in NEWS_TERMS):
        datasets.append("global_news")
    if len(constraints["tickers"]) > 1 and any(
        term in text for term in {"invest", "best bet"}
    ):
        datasets = ["global_news", "stock_price_details", "sec_filings"]
    return datasets or ["global_news", "stock_price_details", "sec_filings"]

# Define a function to extract constraints from the question, such as ticker symbols and dates.
def extract_question_constraints(question):
    """Extract known tickers and explicit ISO dates from a question."""
    text = question.lower()
    tickers = {
        ticker for ticker in KNOWN_TICKERS if ticker.lower() in text
    }
    tickers.update({
        ticker
        for alias, ticker in COMPANY_TICKER_ALIASES.items()
        if alias and alias in text
    })
    dates = re.findall(r"\b\d{4}-\d{2}-\d{2}\b", question)
    return {"tickers": sorted(tickers), "dates": dates}

# Define a function to create a Chroma filter based on the dataset, ticker, and date constraints.
def chroma_filter(dataset, ticker=None, date=None):
    """Build a Chroma metadata filter for one routed dataset search."""
    conditions = [{"dataset": {"$eq": dataset}}]
    if dataset == "stock_price_details" and ticker:
        conditions.append({"ticker": {"$eq": ticker}})
    if dataset == "stock_price_details" and date:
        conditions.append({"date": {"$eq": date}})
    return conditions[0] if len(conditions) == 1 else {"$and": conditions}


In [40]:
# Cell objective: Retrieve routed evidence, deduplicate it, and return diagnostics.
# Define a function to retrieve context from the vector store based on the question, constraints, and search type.
def retrieve_context(question, k=5, search_type="similarity", use_routing=True, store=None):
    """Retrieve, deduplicate, and diagnose source-aware evidence for a question."""
    active_store = store or vector_store
    datasets = route_question(question) if use_routing else ["global_news", "stock_price_details", "sec_filings"]
    constraints = extract_question_constraints(question)
    k_per_dataset = k if len(datasets) == 1 else max(2, k // len(datasets)) # Distribute the top-k results across datasets
    retrieved_docs = []
    searches = []

    # Iterate through each dataset and perform a similarity search or max marginal relevance search based on the specified search type. 
    for dataset in datasets:
        requested_date = constraints["dates"][0] if len(constraints["dates"]) == 1 else None
        ticker_targets = (
            constraints["tickers"]
            if dataset == "stock_price_details" and constraints["tickers"]
            else [None]
        )
        k_per_target = max(1, k_per_dataset // len(ticker_targets))

        for ticker in ticker_targets:
            where = chroma_filter(dataset, ticker=ticker, date=requested_date)
            if search_type == "mmr":
                docs = active_store.max_marginal_relevance_search(
                    question, k=k_per_target,
                    fetch_k=max(20, k_per_target * 4), filter=where
                )
            else:
                docs = active_store.similarity_search(
                    question, k=k_per_target, filter=where
                )

            fallback_used = False
            if not docs and dataset == "stock_price_details" and requested_date:
                fallback_used = True
                where = chroma_filter(dataset, ticker=ticker)
                docs = active_store.similarity_search(
                    question, k=k_per_target, filter=where
                )

            retrieved_docs.extend(docs)
            searches.append({
                "dataset": dataset,
                "ticker": ticker,
                "filter": where,
                "result_count": len(docs),
                "fallback_used": fallback_used,
            })

    unique_docs = []
    seen_ids = set()
    for doc in retrieved_docs:
        doc_id = deterministic_document_id(doc)
        if doc_id not in seen_ids:
            seen_ids.add(doc_id)
            unique_docs.append(doc)

    diagnostics = {
        "datasets_requested": datasets,
        "constraints": constraints,
        "search_type": search_type,
        "searches": searches,
        "retrieved_count": len(unique_docs),
        "retrieved_datasets": sorted({doc.metadata["dataset"] for doc in unique_docs}),
    }
    return {"documents": unique_docs, "diagnostics": diagnostics}


In [41]:
# Cell objective: Configure the answer model and define the baseline RAG function.
# Model rationale: GPT-4.1 mini was selected as a fast and cost-efficient
# generation model with strong instruction-following capabilities. These
# characteristics help the RAG workflow remain grounded in retrieved evidence,
# cite sources, recognize missing information, and support repeated prompt
# optimization and configuration-evaluation calls. Although newer models may
# provide higher capability for production workloads, GPT-4.1 mini offers an
# appropriate balance of quality, latency, cost, and experimental
# reproducibility for this proof of concept. Temperature zero further reduces
# generation variability during controlled comparisons.
baseline_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Define a function to answer a question using the baseline retrieval-augmented generation (RAG) approach.
def baseline_rag_answer(question, k=5):
    """Run the baseline retriever and prompt, returning the answer and its evidence."""
    retrieval_result = retrieve_context(question, k=k)
    retrieved_docs = retrieval_result["documents"]
    context = format_retrieved_context(retrieved_docs)

    user_prompt = f"""
Question:
{question}

Retrieved context:
{context}

Answer with a direct response followed by a short Sources section naming the most relevant source numbers.
""".strip()

    response = baseline_llm.invoke([
        SystemMessage(content=baseline_system_prompt),
        HumanMessage(content=user_prompt),
    ])

    return {
        "question": question,
        "answer": response.content,
        "context": context,
        "source_documents": retrieved_docs,
        "retrieval_diagnostics": retrieval_result["diagnostics"],
    }


In [42]:
# Cell objective: Run one transparent end-to-end baseline example.
sample_question = "What are the main risks or signals a broker should discuss with a client considering Apple?"
baseline_sample = baseline_rag_answer(sample_question)

print("Question:")
print(baseline_sample["question"])

print("\nAnswer:")
print(baseline_sample["answer"])

print("\nBEGIN Context:")
print(baseline_sample["context"])
print("\nEND Context:")

print("\nRetrieved sources:")
for idx, doc in enumerate(baseline_sample["source_documents"], start=1):
    print(f"[{idx}] {format_source_metadata(doc.metadata or {})}")

print("\nRetrieval diagnostics:")    
print(baseline_sample["retrieval_diagnostics"])


Question:
What are the main risks or signals a broker should discuss with a client considering Apple?

Answer:
Key risks and signals to discuss with a client considering Apple include:

- Gross margin volatility and downward pressure, which could impact profitability.
- Operational disruptions and increased costs due to modifications in products, services, and operations.
- Heightened privacy and data security risks that may lead to potential liabilities and fines.
- Exposure to interest rate fluctuations, with the company using hedging instruments to manage this risk.
- Potential adverse effects on business reputation, financial condition, and stock price from the above factors.

These risks are detailed in Apple's Q2 2026 Form 10-Q under the "Risk Factors" section.

Sources: 1, 2, 3, 5

BEGIN Context:
Corpus snapshot date: 2026-04-23

[Source 1] Form 10-Q, in each case under the heading “Risk Factors.” As a result, the Company believes, in general, gross margins will be subject to vo

## 1.5 Load the gold benchmark dataset and evaluate the baseline prompt


In [43]:
# Cell objective: Load the golden benchmark from either supported notebook location.
# Objective: create a leakage-resistant benchmark workflow. GEPA optimization
# and final prompt evaluation use separate, source-balanced subsets. Retrieval
# is executed once and cached so both prompts receive identical evidence.
gold_dataset_path = Path("../data/eval/golden_benchmark_dataset.csv")
if not gold_dataset_path.exists():
    gold_dataset_path = Path("data/eval/golden_benchmark_dataset.csv")

gold_df = pd.read_csv(gold_dataset_path)


In [44]:
# Cell objective: Validate benchmark columns, rows, case IDs, and source labels.
# Validate that the gold benchmark dataset contains all required columns. 
# If any required columns are missing, raise a ValueError with a message indicating which columns are missing. 
# This ensures that the dataset is complete and suitable for evaluation.
required_columns = {"question", "response", "source_hint", "supporting_sources", "context"}
missing_columns = required_columns.difference(gold_df.columns)
if missing_columns:
    raise ValueError(f"Gold benchmark dataset is missing columns: {sorted(missing_columns)}")

# Drop any rows with missing values in the required columns and reset the index. 
# This ensures that the benchmark dataset is clean and ready for evaluation.
gold_df = gold_df.dropna(subset=["question", "response"]).reset_index(drop=True)

# gold_df.index is a RangeIndex starting from 0, so adding 1 to it will create a new column "case_id" that starts from 1.
# This is useful for tracking and referencing individual cases in the benchmark dataset.
gold_df["case_id"] = gold_df.index + 1

# Map the supporting source labels in the gold benchmark dataset to their corresponding expected dataset names.
SOURCE_DATASET_MAP = {
    "global_news.csv": "global_news",
    "all_prices_clean.csv": "stock_price_details",
    "stock_price_details.csv": "stock_price_details",
    "sec_filings.txt": "sec_filings",
    "sec_filings_10q.pdf": "sec_filings",
}
gold_df["expected_dataset"] = gold_df["supporting_sources"].map(SOURCE_DATASET_MAP)
if gold_df["expected_dataset"].isna().any():
    unknown_sources = sorted(gold_df.loc[gold_df["expected_dataset"].isna(), "supporting_sources"].unique())
    raise ValueError(f"Unknown supporting source labels: {unknown_sources}")


In [45]:
# Cell objective: Create a reproducible source-balanced optimization and holdout split.
# Reserve approximately 40% of the cases from each expected source for holdout evaluation.
# The remaining cases are used for GEPA prompt optimization.
# Keeping optimization and evaluation separate prevents overfitting and data leakage:
# - Overfitting: optimizing the prompt for a specific set of cases instead of generalizing.
# - Data leakage: allowing evaluation cases to influence prompt optimization, leading to overly optimistic results.
# Grouping by `expected_dataset` preserves representation from each source, while
# `random_state=42` makes the split reproducible.


evaluation_df = (
    gold_df.groupby("expected_dataset", group_keys=False)
    .sample(frac=0.4, random_state=42)
    .sort_values("case_id")
    .reset_index(drop=True)
)
optimization_df = (
    gold_df[~gold_df["case_id"].isin(evaluation_df["case_id"])]
    .sort_values("case_id")
    .reset_index(drop=True)
)


In [46]:
# Cell objective: Freeze retrieval evidence once for fair prompt comparisons.
# Objective: cache the retrieval results for every gold benchmark question. This
# ensures that both the GEPA prompt optimization and final evaluation use identical
# evidence. The retrieval diagnostics are also cached for later analysis.
benchmark_contexts = {}
for _, row in gold_df.iterrows():
    retrieval_result = retrieve_context(row["question"], k=5)
    docs = retrieval_result["documents"]
    benchmark_contexts[row["question"]] = {
        "documents": docs,
        "formatted_context": format_retrieved_context(docs),
        "retrieval_context": [doc.page_content for doc in docs],
        "retrieved_sources": [format_source_metadata(doc.metadata or {}) for doc in docs],
        "diagnostics": retrieval_result["diagnostics"],
    }


In [47]:
# Cell objective: Define deterministic retrieval coverage checks.
# Define a function to evaluate the quality of the retrieval results against the expected dataset, ticker, and date constraints.
def retrieval_quality(row, cached_context):
    """Check whether retrieved documents cover the expected source and constraints."""
    docs = cached_context["documents"]
    retrieved_datasets = sorted({doc.metadata["dataset"] for doc in docs})
    expected_dataset = row["expected_dataset"]
    constraints = extract_question_constraints(row["question"])
    source_match = expected_dataset in retrieved_datasets
    ticker_match = (
        None
        if expected_dataset != "stock_price_details" or not constraints["tickers"]
        else all(
            any(doc.metadata.get("ticker") == ticker for doc in docs)
            for ticker in constraints["tickers"]
        )
    )
    requested_dates = constraints["dates"]
    date_match = (
        None if expected_dataset != "stock_price_details" or not requested_dates else
        all(any(doc.metadata.get("date") == date for doc in docs) for date in requested_dates)
    )
    if not source_match:
        status = "UNSUPPORTED"
    elif ticker_match is False or date_match is False:
        status = "PARTIAL"
    else:
        status = "SUPPORTED"
    return {
        "expected_dataset": expected_dataset,
        "retrieved_datasets": retrieved_datasets,
        "source_match": source_match,
        "ticker_match": ticker_match,
        "date_match": date_match,
        "unique_sources": len(set(cached_context["retrieved_sources"])),
        "coverage_status": status,
    }


In [50]:
# Cell objective: Generate baseline answers and DeepEval test cases on the holdout split.
# Objective: generate baseline outputs for every holdout evaluation case. The
# baseline RAG approach uses the same retrieval evidence as the GEPA prompt optimization.
baseline_eval_df = evaluation_df.copy()


baseline_rows = []
baseline_test_cases = []

# Iterate through each row in the baseline evaluation DataFrame and generate a response using the baseline RAG approach.
for _, row in baseline_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["response"]
    cached = benchmark_contexts[question]
    retrieval_checks = retrieval_quality(row, cached)

    user_prompt = f"""
Question:
{question}

Retrieved context:
{cached['formatted_context']}

Answer with a direct response followed by a short Sources section naming the most relevant source numbers.
""".strip()
    
    # Invoke the baseline LLM with the system prompt and user prompt to generate a response.
    response = baseline_llm.invoke([
        SystemMessage(content=baseline_system_prompt),
        HumanMessage(content=user_prompt),
    ])

    # Store the results in a dictionary and append it to the baseline_rows list for later analysis.
    baseline_rows.append({
        "case_id": int(row["case_id"]),
        "question": question,
        "expected_output": expected_answer,
        "actual_output": response.content,
        "retrieved_context": cached["formatted_context"],
        "retrieved_sources": cached["retrieved_sources"],
        "source_hint": row["source_hint"],
        "supporting_sources": row["supporting_sources"],
        **retrieval_checks,
    })

    # Create an LLMTestCase object for the current question and append it to the baseline_test_cases list for later evaluation.
    baseline_test_cases.append(
        LLMTestCase(
            input=question,
            actual_output=response.content,
            expected_output=expected_answer,
            retrieval_context=cached["retrieval_context"],
        )
    )

# Create a DataFrame from the baseline_rows list for easier analysis and display.
baseline_results_df = pd.DataFrame(baseline_rows)


In [51]:
# Cell objective: Display the benchmark split and retrieval-coverage summary.
print(f"Loaded {len(gold_df)} gold benchmark examples from {gold_dataset_path.resolve()}")
print(f"Optimization cases: {len(optimization_df)} | Holdout evaluation cases: {len(evaluation_df)}")
print(f"Generated baseline outputs for {len(baseline_results_df)} holdout examples")
display(baseline_results_df[["case_id", "expected_dataset", "retrieved_datasets", "coverage_status", "source_match"]])


Loaded 20 gold benchmark examples from /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/data/eval/golden_benchmark_dataset.csv
Optimization cases: 12 | Holdout evaluation cases: 8
Generated baseline outputs for 8 holdout examples


,case_id,expected_dataset,retrieved_datasets,coverage_status,source_match
0,1,sec_filings,[sec_filings],SUPPORTED,True
1,4,sec_filings,[sec_filings],SUPPORTED,True
2,7,global_news,[global_news],SUPPORTED,True
3,8,global_news,[global_news],SUPPORTED,True
4,13,stock_price_details,[stock_price_details],SUPPORTED,True
5,16,global_news,[global_news],SUPPORTED,True
6,17,sec_filings,"[global_news, sec_filings, stock_price_details]",SUPPORTED,True
7,20,sec_filings,"[sec_filings, stock_price_details]",SUPPORTED,True


## 1.6 Define the evaluation metrics

In [52]:
# Cell objective: Configure the shared threshold and LLM-as-a-judge model.
EVALUATION_THRESHOLD = 0.5
EVALUATOR_MODEL_NAME = "gpt-4.1-mini"

# GPT-4.1 mini balances instruction-following quality, latency, and cost for
# repeated LLM-as-a-judge calls. Production use should add human review or an
# independent judge model to detect correlated scoring bias.
evaluation_model = GPTModel(model=EVALUATOR_MODEL_NAME)
print(f"DeepEval evaluator model: {EVALUATOR_MODEL_NAME}")


DeepEval evaluator model: gpt-4.1-mini


In [53]:
# Cell objective: Define a reusable factory for consistently configured GEval metrics.
def make_geval(name, criteria, evaluation_params):
    """Create one synchronous GEval metric with shared model and threshold settings."""
    return GEval(
        name=name,
        threshold=EVALUATION_THRESHOLD,
        model=evaluation_model,
        criteria=criteria,
        evaluation_params=evaluation_params,
        async_mode=False,
    )


In [54]:
# Cell objective: Define reference-based Answer Correctness.
correctness_metric = make_geval(
    name="Answer Correctness",
    criteria=(
        "Evaluate whether the actual output correctly answers the input question "
        "and matches the expected output from the gold benchmark dataset."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
)


In [55]:
# Cell objective: Define Faithfulness / Groundedness against retrieved evidence.
faithfulness_metric = make_geval(
    name="Faithfulness / Groundedness",
    criteria=(
        "Evaluate whether the actual output is fully supported by the retrieved context. "
        "Penalize unsupported claims, hallucinations, or facts not present in the context."
    ),
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
)


In [56]:
# Cell objective: Define retriever-focused Context Relevance.
context_relevance_metric = make_geval(
    name="Context Relevance",
    criteria=(
        "Evaluate whether the retrieved context is relevant and sufficient to answer "
        "the input question."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
)


In [57]:
# Cell objective: Define direct-answer relevance.
answer_relevance_metric = make_geval(
    name="Answer Relevance",
    criteria=(
        "Evaluate whether the actual output directly answers the input question "
        "without unnecessary or unrelated information."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
    ],
)


In [58]:
# Cell objective: Define client-facing Broker Actionability.
broker_actionability_metric = make_geval(
    name="Broker Actionability",
    criteria=(
        "Evaluate whether the answer is useful for a broker speaking with a client. "
        "It should highlight relevant risks, signals, evidence, and implications."
    ),
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.RETRIEVAL_CONTEXT,
    ],
)


In [59]:
# Cell objective: Group metrics according to what each experimental stage can validly measure.
# Context Relevance evaluates the retriever and is measured separately when
# baseline and optimized prompts share the same frozen evidence.
retrieval_metrics = [context_relevance_metric]
prompt_evaluation_metrics = [
    correctness_metric,
    faithfulness_metric,
    answer_relevance_metric,
    broker_actionability_metric,
]
rag_evaluation_metrics = retrieval_metrics + prompt_evaluation_metrics

# Official questions have no human-written expected answers, so correctness is
# excluded instead of claiming a reference-based score that cannot be computed.
official_test_evaluation_metrics = [
    context_relevance_metric,
    faithfulness_metric,
    answer_relevance_metric,
    broker_actionability_metric,
]
prompt_optimization_metrics = prompt_evaluation_metrics.copy()


## 1.7 Evaluate the baseline RAG on the golden examples dataset

In [60]:
# Cell objective: Define safe, reusable measurement for one DeepEval metric.
def measure_metric(metric, test_case):
    """Measure one DeepEval metric and return a consistent result record."""
    try:
        metric.measure(test_case, _show_indicator=False)
        return {
            "score": metric.score,
            "success": metric.is_successful() if hasattr(metric, "is_successful") else None,
            "reason": metric.reason,
            "error": None,
        }
    except Exception as exc:
        return {"score": None, "success": False, "reason": None, "error": str(exc)}


In [61]:
# Cell objective: Define reusable aggregation for long-form metric results.
def summarize_metrics(evaluation_df):
    """Aggregate long-form metric results into a compact score table."""
    return (
        evaluation_df.groupby("metric", dropna=False)
        .agg(
            average_score=("score", "mean"),
            evaluated_cases=("score", "count"),
            successful_cases=("success", "sum"),
            failed_or_error_cases=("error", lambda values: values.notna().sum()),
        )
        .reset_index()
        .sort_values("average_score", ascending=False)
    )


In [62]:
# Cell objective: Define one evaluator shared by baseline and optimized answers.
def evaluate_test_cases(result_rows, test_cases, metrics, label):
    """Evaluate a collection of test cases with the same metric suite."""
    evaluation_rows = []
    for position, (result_row, test_case) in enumerate(zip(result_rows, test_cases), start=1):
        print(f"Evaluating {label} case {result_row['case_id']} ({position}/{len(test_cases)})")
        for metric in metrics:
            evaluation_rows.append({
                "case_id": result_row["case_id"],
                "question": test_case.input,
                "metric": metric.name,
                "actual_output": test_case.actual_output,
                "expected_output": test_case.expected_output,
                "coverage_status": result_row["coverage_status"],
                "source_match": result_row["source_match"],
                **measure_metric(metric, test_case),
            })
    return pd.DataFrame(evaluation_rows)


In [64]:
# Cell objective: Evaluate frozen retrieval independently from prompt quality.
# Retrieval is evaluated once because the baseline and optimized prompts use
# exactly the same frozen question-context pairs.
retrieval_evaluation_df = evaluate_test_cases(
    baseline_rows,
    baseline_test_cases,
    retrieval_metrics,
    label="retrieval",
)
retrieval_metric_summary_df = summarize_metrics(retrieval_evaluation_df)


Evaluating retrieval case 1 (1/8)
Evaluating retrieval case 4 (2/8)
Evaluating retrieval case 7 (3/8)
Evaluating retrieval case 8 (4/8)
Evaluating retrieval case 13 (5/8)
Evaluating retrieval case 16 (6/8)
Evaluating retrieval case 17 (7/8)
Evaluating retrieval case 20 (8/8)


In [65]:
# Cell objective: Evaluate baseline answers with prompt-controlled metrics.
baseline_evaluation_df = evaluate_test_cases(
    baseline_rows,
    baseline_test_cases,
    prompt_evaluation_metrics,
    label="baseline prompt",
)
baseline_metric_summary_df = summarize_metrics(baseline_evaluation_df)
baseline_scores_pivot_df = baseline_evaluation_df.pivot(
    index="case_id", columns="metric", values="score"
).reset_index()


Evaluating baseline prompt case 1 (1/8)
Evaluating baseline prompt case 4 (2/8)
Evaluating baseline prompt case 7 (3/8)
Evaluating baseline prompt case 8 (4/8)
Evaluating baseline prompt case 13 (5/8)
Evaluating baseline prompt case 16 (6/8)
Evaluating baseline prompt case 17 (7/8)
Evaluating baseline prompt case 20 (8/8)


In [66]:
# Cell objective: Display compact baseline retrieval, answer, and coverage summaries.
print("Frozen-context retrieval evaluation")
display(retrieval_metric_summary_df)
print("Baseline prompt evaluation")
display(baseline_metric_summary_df)
display(baseline_scores_pivot_df)
print("Deterministic retrieval coverage")
display(
    baseline_results_df["coverage_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="cases")
)


Frozen-context retrieval evaluation


,metric,average_score,evaluated_cases,successful_cases,failed_or_error_cases
0,Context Relevance,0.453936,8,3,0


Baseline prompt evaluation


,metric,average_score,evaluated_cases,successful_cases,failed_or_error_cases
1,Answer Relevance,0.881271,8,8,0
3,Faithfulness / Groundedness,0.862415,8,8,0
2,Broker Actionability,0.858241,8,8,0
0,Answer Correctness,0.662184,8,6,0


metric,case_id,Answer Correctness,Answer Relevance,Broker Actionability,Faithfulness / Groundedness
0,1,0.798594,0.902931,0.903733,0.816482
1,4,0.802946,0.902298,0.900000,0.780290
2,7,0.629428,0.898901,0.826894,0.873106
3,8,0.901099,0.901406,0.850000,0.820964
4,13,0.277730,0.832082,0.798730,0.904743
5,16,0.900000,0.900000,0.900000,0.900000
6,17,0.792414,0.900000,0.900000,0.900000
7,20,0.195257,0.812550,0.786574,0.903733


Deterministic retrieval coverage


,status,cases
0,SUPPORTED,8


# Section 2: DeepEval Prompt Optimization Layer

## 2.1 Define the prompt template and model callback


In [67]:
# Cell objective: Define the seed prompt that GEPA will improve.
# Objective: optimize only the prompt. Every candidate receives a frozen
# cached context, so retrieval cannot confound the GEPA comparison.
optimized_prompt_seed = Prompt(
    text_template="""
You are a grounded investment intelligence assistant for brokerage analysts.

Use only the retrieved context to answer the question. Do not invent facts.
If the context does not contain enough evidence, clearly state what is missing.
Write a concise client-ready answer that explains the relevant financial signal,
risk, or implication. Reference the most relevant source numbers when useful.

Question:
{question}

Retrieved context:
{context}

Answer:
""".strip()
)
seed_prompt_text = optimized_prompt_seed.text_template


In [68]:
# Cell objective: Convert optimization rows into DeepEval Golden examples.
prompt_optimization_goldens = [
    Golden(
        input=row["question"],
        expected_output=row["response"],
        additional_metadata={
            "source_hint": row["source_hint"],
            "supporting_sources": row["supporting_sources"],
            "case_id": int(row["case_id"]),
        },
    )
    for _, row in optimization_df.iterrows()
]


In [69]:
# Cell objective: Define the GEPA callback and verify the optimization inputs.
def optimized_prompt_model_callback(prompt, golden):
    """Render and execute one GEPA prompt candidate against frozen evidence."""
    question = golden.input
    cached = benchmark_contexts[question]
    golden.retrieval_context = cached["retrieval_context"]

    rendered_prompt = prompt.interpolate(
        question=question,
        context=cached["formatted_context"],
    )

    llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    response = llm.invoke([HumanMessage(content=rendered_prompt)])

    return response.content


print(f"Prepared {len(prompt_optimization_goldens)} goldens for prompt optimization")
print("Seed prompt preview:")
print(optimized_prompt_seed.text_template[:700])


Prepared 12 goldens for prompt optimization
Seed prompt preview:
You are a grounded investment intelligence assistant for brokerage analysts.

Use only the retrieved context to answer the question. Do not invent facts.
If the context does not contain enough evidence, clearly state what is missing.
Write a concise client-ready answer that explains the relevant financial signal,
risk, or implication. Reference the most relevant source numbers when useful.

Question:
{question}

Retrieved context:
{context}

Answer:


## 2.2 Optimize the prompt with GEPA

In [70]:
# Cell objective: Import GEPA support classes and report the active DeepEval version.
# Objective: run a low-cost, reproducible GEPA optimization using only the
# optimization split. The holdout cases remain unseen until Section 2.3.
from deepeval.optimizer.configs import AsyncConfig, DisplayConfig
from deepeval.optimizer.scorer.scorer import Scorer
from deepeval.optimizer.rewriter.rewriter import Rewriter

print(f"DeepEval version: {getattr(deepeval, '__version__', 'unknown')}")


DeepEval version: 4.0.9


In [71]:
# Cell objective: Apply the version-specific token-accounting compatibility patch.
# Workaround for a DeepEval GEPA token-accounting bug in this installed version.
# GEPA calls _accrue_tokens on helper classes that do not define it.
def _optimizer_helper_accrue_tokens(self, input_tokens=0, output_tokens=0):
    """Patch token accounting for DeepEval helper classes when required."""
    self.input_tokens = getattr(self, "input_tokens", 0) + (input_tokens or 0)
    self.output_tokens = getattr(self, "output_tokens", 0) + (output_tokens or 0)


for helper_cls in (Scorer, Rewriter):
    if not hasattr(helper_cls, "_accrue_tokens"):
        helper_cls._accrue_tokens = _optimizer_helper_accrue_tokens


In [72]:
# Cell objective: Configure GEPA and the prompt optimizer reproducibly.
gepa_algorithm = GEPA(
    iterations=3,
    minibatch_size=8,
    pareto_size=5,
    patience=2,
    random_seed=42,
    tie_breaker=TieBreaker.PREFER_CHILD,
    reflection_model="gpt-4.1-mini",
    mutation_model="gpt-4.1-mini",
)

prompt_optimizer = PromptOptimizer(
    model_callback=optimized_prompt_model_callback,
    metrics=prompt_optimization_metrics,
    optimizer_model="gpt-4.1-mini",
    algorithm=gepa_algorithm,
    async_config=AsyncConfig(run_async=False, max_concurrent=1, throttle_value=0.5),
    display_config=DisplayConfig(show_indicator=False),
)


In [73]:
# Cell objective: Optimize the seed prompt on the optimization split only.
optimized_prompt = prompt_optimizer.optimize(
    prompt=optimized_prompt_seed,
    goldens=prompt_optimization_goldens,
)

optimization_report = prompt_optimizer.optimization_report
optimized_prompt_text = optimized_prompt.text_template

print("Optimized prompt:")
print(optimized_prompt_text)


Optimized prompt:
You are a grounded investment intelligence assistant for brokerage analysts.

Use only the retrieved context to answer the question. Do not invent facts.
If the context does not contain enough evidence, clearly state what is missing.
Write a concise client-ready answer that explains the relevant financial signal,
risk, or implication. Reference the most relevant source numbers when useful.

Question:
{question}

Retrieved context:
{context}

Answer:


## 2.3 Evaluate the optimized prompt on the benchmark dataset

In [74]:
# Cell objective: Configure deterministic generation for the optimized prompt.
# Objective: evaluate the optimized prompt only on the holdout split. The
# cached baseline contexts are reused exactly, isolating the prompt as the
# single experimental variable.
optimized_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


In [75]:
# Cell objective: Generate optimized answers on the untouched prompt holdout.
# Generate answers with the optimized prompt using the same benchmark and
# retrieval settings used for the baseline evaluation.
optimized_rows = []
optimized_test_cases = []

for _, row in evaluation_df.iterrows():
    question = row["question"]
    expected_answer = row["response"]
    cached = benchmark_contexts[question]
    retrieval_checks = retrieval_quality(row, cached)

    rendered_prompt = optimized_prompt.interpolate(
        question=question,
        context=cached["formatted_context"],
    )

    response = optimized_llm.invoke([
        HumanMessage(content=rendered_prompt)
    ])
    actual_answer = response.content

    optimized_rows.append({
        "case_id": int(row["case_id"]),
        "question": question,
        "expected_output": expected_answer,
        "actual_output": actual_answer,
        "retrieved_context": cached["formatted_context"],
        "retrieved_sources": cached["retrieved_sources"],
        "source_hint": row["source_hint"],
        "supporting_sources": row["supporting_sources"],
        **retrieval_checks,
    })

    optimized_test_cases.append(
        LLMTestCase(
            input=question,
            actual_output=actual_answer,
            expected_output=expected_answer,
            retrieval_context=cached["retrieval_context"],
        )
    )

optimized_results_df = pd.DataFrame(optimized_rows)

print(f"Generated optimized-prompt outputs for {len(optimized_results_df)} examples")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 300)
display(
    optimized_results_df[
        ["question", "expected_output", "actual_output", "supporting_sources"]
    ]
)


Generated optimized-prompt outputs for 8 examples


,question,expected_output,actual_output,supporting_sources
0,"Given the internal control audit for the fiscal year ended September 27, 2025, did Apple’s auditors identify any material weaknesses, and how does this affect the reliability of the current financial disclosures?","No material weaknesses were identified. Ernst & Young LLP issued an unqualified opinion stating that Apple maintained effective internal control over financial reporting as of September 27, 2025 based on the COSO 2013 framework. This supports a high level of reliability for the company’s financi...","Apple’s auditors did not identify any material weaknesses in internal control over financial reporting for the fiscal year ended September 27, 2025. According to management’s evaluation and disclosures in subsequent filings through early 2026, there were no changes or deficiencies that materiall...",sec_filings.txt
1,A client is skeptical of automated financial reporting. How does Apple define Internal Control Over Financial Reporting and what limitation does management acknowledge?,"Apple states that internal controls are designed to provide reasonable assurance regarding financial reporting reliability. Management also acknowledges that internal controls cannot prevent or detect all errors and fraud, and therefore provide reasonable rather than absolute assurance.","Apple defines Internal Control Over Financial Reporting (ICFR) as a system of disclosure controls and procedures designed to provide reasonable assurance that information required to be disclosed in SEC reports is recorded, processed, summarized, and reported within the required time periods, an...",sec_filings.txt
2,SRx Health Solutions recently announced a pivot toward the convergence of AI and space. What is the scale of the investment and who is their digital-asset partner?,SRx Health Solutions invested more than 10 percent of its investable capital into Astro Investment XVII and partnered with EMJ Crypto Technologies as the digital-asset treasury operating platform.,"SRx Health Solutions has committed over 10% of its investable capital toward the convergence of Artificial Intelligence (AI) and space through an investment in Astro Investment XVII, an affiliate of Astro Capital. Their digital-asset partner in this initiative is EMJ Crypto Technologies, which o...",global_news.csv
3,"For brokers monitoring small-cap AI plays, what exchange and ticker symbol should be tracked for SRx Health Solutions?",SRx Health Solutions trades on the NYSE American exchange under the ticker SRXH.,"Brokers monitoring small-cap AI plays should track SRx Health Solutions on the NYSE American exchange under the ticker symbol **SRXH**. This is confirmed by the April 22, 2026 GlobeNewswire report stating SRx Health Solutions, Inc. trades as NYSE American: SRXH.",global_news.csv
4,"Compare Tencent’s closing price on October 23, 2025 and November 6, 2025. Did trading volume support a strong directional breakout?","Tencent’s closing price increased from 633.0 to 644.0 between October 23 and November 6, 2025. Trading volume remained relatively stable, suggesting a gradual upward move rather than a strong breakout.","The available data includes Tencent’s closing price and volume on October 23, 2025 (Close: 633.0 HKD, Volume: 12,419,348 shares) but does not provide any trading data for November 6, 2025. Without the closing price and volume for November 6, 2025, it is not possible to compare the two dates or a...",all_prices_clean.csv
5,"How does the April 17, 2026 market intelligence reporting connect the Strait of Hormuz blockade to Asian fuel supply concerns?","The reports explain that the naval blockade disrupted Middle Eastern energy flows, forcing Asian refiners in Japan and South Korea to source additional US crude oil supplies in order to avoid fuel shortages.","The April 17, 2026 market intelligence highlights a direct connection between the Strait of Hormuz blockade and Asian fuel supply concerns. Specifically, 

In [76]:
# Cell objective: Evaluate optimized answers with the same baseline metric suite.
# Evaluate with the exact same helper and metric suite used for the baseline.
optimized_evaluation_df = evaluate_test_cases(
    optimized_rows,
    optimized_test_cases,
    prompt_evaluation_metrics,
    label="optimized prompt",
)


Evaluating optimized prompt case 1 (1/8)
Evaluating optimized prompt case 4 (2/8)
Evaluating optimized prompt case 7 (3/8)
Evaluating optimized prompt case 8 (4/8)
Evaluating optimized prompt case 13 (5/8)
Evaluating optimized prompt case 16 (6/8)
Evaluating optimized prompt case 17 (7/8)
Evaluating optimized prompt case 20 (8/8)


In [77]:
# Cell objective: Display compact optimized-prompt summaries by metric and case.
optimized_metric_summary_df = summarize_metrics(optimized_evaluation_df)

print("Optimized prompt evaluation summary")
display(optimized_metric_summary_df)

# Present one row per benchmark case, with each metric as a column.
# The full long-form results remain available in optimized_evaluation_df.
optimized_scores_pivot_df = optimized_evaluation_df.pivot(
    index="case_id",
    columns="metric",
    values="score",
).reset_index()

print("Optimized prompt scores by benchmark case")
display(optimized_scores_pivot_df)


Optimized prompt evaluation summary


,metric,average_score,evaluated_cases,successful_cases,failed_or_error_cases
2,Broker Actionability,0.894465,8,8,0
1,Answer Relevance,0.887823,8,8,0
3,Faithfulness / Groundedness,0.859856,8,8,0
0,Answer Correctness,0.700063,8,6,0


Optimized prompt scores by benchmark case


metric,case_id,Answer Correctness,Answer Relevance,Broker Actionability,Faithfulness / Groundedness
0,1,0.793991,0.902298,0.967918,0.733081
1,4,0.814805,0.877730,0.900000,0.711883
2,7,0.900000,0.900000,0.900000,0.890465
3,8,0.922270,0.867918,0.898201,0.909535
4,13,0.243782,0.892414,0.877730,0.932082
5,16,0.901406,0.906009,0.900000,0.900000
6,17,0.806009,0.900000,0.903733,0.901799
7,20,0.218243,0.856218,0.808139,0.900000


## 2.4 Compare optimized prompt with the baseline prompt


In [78]:
# Cell objective: Merge baseline and optimized metric summaries for direct comparison.
# Objective: compare prompts at metric, case, and overall levels, then apply
# quality gates so a small average gain cannot hide a critical regression.
baseline_comparison_df = baseline_metric_summary_df.rename(
    columns={
        "average_score": "baseline_average_score",
        "evaluated_cases": "baseline_evaluated_cases",
        "successful_cases": "baseline_successful_cases",
        "failed_or_error_cases": "baseline_error_cases",
    }
)

optimized_comparison_df = optimized_metric_summary_df.rename(
    columns={
        "average_score": "optimized_average_score",
        "evaluated_cases": "optimized_evaluated_cases",
        "successful_cases": "optimized_successful_cases",
        "failed_or_error_cases": "optimized_error_cases",
    }
)

prompt_comparison_df = (
    baseline_comparison_df
    .merge(optimized_comparison_df, on="metric", how="outer")
)

prompt_comparison_df["score_delta"] = (
    prompt_comparison_df["optimized_average_score"]
    - prompt_comparison_df["baseline_average_score"]
)

nonzero_baseline_scores = prompt_comparison_df[
    "baseline_average_score"
].replace(0, pd.NA)

prompt_comparison_df["improvement_pct"] = (
    prompt_comparison_df["score_delta"]
    .div(nonzero_baseline_scores)
    .mul(100)
)


In [79]:
# Cell objective: Assign a winner for each metric and display score deltas.
def select_winner(row, tolerance=1e-9):
    """Compare baseline and optimized scores while allowing numerical ties."""
    baseline_score = row["baseline_average_score"]
    optimized_score = row["optimized_average_score"]

    if pd.isna(baseline_score) or pd.isna(optimized_score):
        return "Unavailable"
    if abs(optimized_score - baseline_score) <= tolerance:
        return "Tie"
    if optimized_score > baseline_score:
        return "Optimized"
    return "Baseline"


prompt_comparison_df["winner"] = prompt_comparison_df.apply(
    select_winner,
    axis=1,
)

prompt_comparison_df = prompt_comparison_df.sort_values(
    "score_delta",
    ascending=False,
).reset_index(drop=True)

print("Baseline vs. optimized prompt — metric summary")
display(
    prompt_comparison_df[
        [
            "metric",
            "baseline_average_score",
            "optimized_average_score",
            "score_delta",
            "improvement_pct",
            "winner",
            "baseline_error_cases",
            "optimized_error_cases",
        ]
    ].round(4)
)


Baseline vs. optimized prompt — metric summary


,metric,baseline_average_score,optimized_average_score,score_delta,improvement_pct,winner,baseline_error_cases,optimized_error_cases
0,Answer Correctness,0.6622,0.7001,0.0379,5.7204,Optimized,0,0
1,Broker Actionability,0.8582,0.8945,0.0362,4.2207,Optimized,0,0
2,Answer Relevance,0.8813,0.8878,0.0066,0.7435,Optimized,0,0
3,Faithfulness / Groundedness,0.8624,0.8599,-0.0026,-0.2967,Baseline,0,0


In [80]:
# Cell objective: Compare both prompts at the individual case-and-metric level.
# Compare both prompts for each benchmark case and metric.
baseline_case_scores_df = baseline_evaluation_df[
    ["case_id", "question", "metric", "score"]
].rename(columns={"score": "baseline_score"})

optimized_case_scores_df = optimized_evaluation_df[
    ["case_id", "question", "metric", "score"]
].rename(columns={"score": "optimized_score"})

case_level_comparison_df = baseline_case_scores_df.merge(
    optimized_case_scores_df,
    on=["case_id", "question", "metric"],
    how="inner",
)

case_level_comparison_df["score_delta"] = (
    case_level_comparison_df["optimized_score"]
    - case_level_comparison_df["baseline_score"]
)

case_level_comparison_df["outcome"] = case_level_comparison_df.apply(
    lambda row: select_winner(
        pd.Series({
            "baseline_average_score": row["baseline_score"],
            "optimized_average_score": row["optimized_score"],
        })
    ),
    axis=1,
)

case_level_outcomes_df = (
    case_level_comparison_df
    .groupby(["metric", "outcome"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("Per-case comparison counts")
display(case_level_outcomes_df)


Per-case comparison counts


outcome,metric,Baseline,Optimized,Tie
0,Answer Correctness,2,6,0
1,Answer Relevance,3,4,1
2,Broker Actionability,0,6,2
3,Faithfulness / Groundedness,3,4,1


In [81]:
# Cell objective: Summarize prompt performance across all evaluated scores.
# Create one compact overall comparison across all questions and metrics.
def evaluation_success_rate(evaluation_df):
    """Calculate the pass rate from non-null metric outcomes."""
    valid_success_values = evaluation_df["success"].dropna()
    if valid_success_values.empty:
        return None
    return valid_success_values.astype(bool).mean()


overall_prompt_comparison_df = pd.DataFrame([
    {
        "prompt": "Baseline",
        "average_score": baseline_evaluation_df["score"].mean(),
        "success_rate": evaluation_success_rate(
            baseline_evaluation_df
        ),
        "evaluated_scores": baseline_evaluation_df["score"].count(),
        "error_count": baseline_evaluation_df["error"].notna().sum(),
    },
    {
        "prompt": "Optimized",
        "average_score": optimized_evaluation_df["score"].mean(),
        "success_rate": evaluation_success_rate(
            optimized_evaluation_df
        ),
        "evaluated_scores": optimized_evaluation_df["score"].count(),
        "error_count": optimized_evaluation_df["error"].notna().sum(),
    },
])

print("Overall prompt comparison")
display(overall_prompt_comparison_df.round(4))


Overall prompt comparison


,prompt,average_score,success_rate,evaluated_scores,error_count
0,Baseline,0.8160,0.9375,32,0
1,Optimized,0.8356,0.9375,32,0


In [82]:
# Cell objective: Verify that GEPA produced a materially different candidate prompt.
def normalize_prompt(text):
    """Normalize whitespace so prompt changes can be compared reliably."""
    return " ".join(text.split())

prompt_was_modified = (
    normalize_prompt(optimized_prompt_text)
    != normalize_prompt(seed_prompt_text)
)
optimized_metric_wins = (
    prompt_comparison_df["winner"] == "Optimized"
).sum()
baseline_metric_wins = (
    prompt_comparison_df["winner"] == "Baseline"
).sum()
metric_ties = (prompt_comparison_df["winner"] == "Tie").sum()

print(f"Did GEPA modify the seed prompt? {prompt_was_modified}")
print(
    "Metric wins — "
    f"Optimized: {optimized_metric_wins}, "
    f"Baseline: {baseline_metric_wins}, "
    f"Ties: {metric_ties}"
)


Did GEPA modify the seed prompt? False
Metric wins — Optimized: 3, Baseline: 1, Ties: 0


In [83]:
# Cell objective: Apply weighted scoring and quality gates before approving the prompt.
# Weighted scoring includes only metrics the generation prompt can influence.
# Context Relevance is reported separately in the retrieval evaluation.
METRIC_WEIGHTS = {
    "Answer Correctness": 0.40,
    "Faithfulness / Groundedness": 0.30,
    "Answer Relevance": 0.10,
    "Broker Actionability": 0.20,
}

def weighted_metric_score(summary_df):
    """Combine prompt-controlled metric averages using project priorities."""
    scores = summary_df.set_index("metric")["average_score"]
    return sum(scores.get(metric, 0) * weight for metric, weight in METRIC_WEIGHTS.items())

baseline_weighted = weighted_metric_score(baseline_metric_summary_df)
optimized_weighted = weighted_metric_score(optimized_metric_summary_df)
baseline_scores = baseline_metric_summary_df.set_index("metric")["average_score"]
optimized_scores = optimized_metric_summary_df.set_index("metric")["average_score"]
baseline_success_rate = evaluation_success_rate(baseline_evaluation_df)
optimized_success_rate = evaluation_success_rate(optimized_evaluation_df)

QUALITY_POLICY = {"minimum_gain": 0.01, "critical_tolerance": 0.02}
quality_gates = {
    "weighted_score_gain": optimized_weighted >= baseline_weighted + QUALITY_POLICY["minimum_gain"],
    "correctness_non_regression": optimized_scores["Answer Correctness"] >= baseline_scores["Answer Correctness"] - QUALITY_POLICY["critical_tolerance"],
    "faithfulness_non_regression": optimized_scores["Faithfulness / Groundedness"] >= baseline_scores["Faithfulness / Groundedness"] - QUALITY_POLICY["critical_tolerance"],
    "success_rate_non_regression": optimized_success_rate >= baseline_success_rate,
    "no_additional_errors": optimized_evaluation_df["error"].notna().sum() <= baseline_evaluation_df["error"].notna().sum(),
}
prompt_decision = "APPROVE" if all(quality_gates.values()) else "REJECT"
quality_gate_df = pd.DataFrame([
    {"gate": gate, "passed": passed}
    for gate, passed in quality_gates.items()
])

print(f"Weighted score — Baseline: {baseline_weighted:.4f} | Optimized: {optimized_weighted:.4f}")
print(f"Final prompt decision: {prompt_decision}")
display(quality_gate_df)


Weighted score — Baseline: 0.7834 | Optimized: 0.8057
Final prompt decision: APPROVE


,gate,passed
0,weighted_score_gain,True
1,correctness_non_regression,True
2,faithfulness_non_regression,True
3,success_rate_non_regression,True
4,no_additional_errors,True


# Section 3: RAG Configuration Tuning and Evaluation


## 3.1 Define the RAG configurations

In [84]:
# Cell objective: Define three named, hypothesis-driven RAG configurations.
# Objective: compare a small set of complete RAG configurations. Chunking
# parameters require a new index, while retrieval and generation parameters
# are applied at runtime. These are holistic candidates for selecting an
# overall approach, not a one-variable-at-a-time causal experiment.
rag_configurations = [
    {
        "name": "Config_2_Balanced",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
        "search_type": "similarity",
        "use_routing": True,
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 500,
        "rationale": "Reference balance between retrieval recall, noise, and response length.",
    },
    {
        "name": "Config_1_Precision_Focused",
        "chunk_size": 700,
        "chunk_overlap": 100,
        "k": 4,
        "search_type": "similarity",
        "use_routing": True,
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 450,
        "rationale": "Tests whether narrower retrieval reduces noise for focused questions.",
    },
    {
        "name": "Config_3_Context_Heavy",
        "chunk_size": 1200,
        "chunk_overlap": 250,
        "k": 7,
        "search_type": "mmr",
        "use_routing": True,
        "temperature": 0.1,
        "top_p": 0.9,
        "max_tokens": 650,
        "rationale": "Tests broader and more diverse evidence for multi-part questions.",
    },
]

rag_configurations = sorted(rag_configurations, key=lambda config: config["name"])


In [85]:
# Cell objective: Select the approved prompt and display the tuning configuration table.
# Use the optimized prompt only when it passed the Section 2.4 quality gates.
selected_prompt_for_tuning = (
    optimized_prompt if prompt_decision == "APPROVE" else optimized_prompt_seed
)
selected_prompt_name = (
    "optimized" if prompt_decision == "APPROVE" else "seed"
)

configuration_table_df = pd.DataFrame(rag_configurations)
print(f"Prompt used for RAG tuning: {selected_prompt_name}")
display(configuration_table_df)


Prompt used for RAG tuning: optimized


,name,chunk_size,chunk_overlap,k,search_type,use_routing,temperature,top_p,max_tokens,rationale
0,Config_1_Precision_Focused,700,100,4,similarity,True,0.0,1.0,450,Tests whether narrower retrieval reduces noise for focused questions.
1,Config_2_Balanced,1000,200,5,similarity,True,0.0,1.0,500,"Reference balance between retrieval recall, noise, and response length."
2,Config_3_Context_Heavy,1200,250,7,mmr,True,0.1,0.9,650,Tests broader and more diverse evidence for multi-part questions.


## 3.2 Create a runtime answering function

In [86]:
# Cell objective: Initialize caches for configuration-specific vector indexes.
# Objective: build the index required by a configuration and answer one
# question with its retrieval and generation settings. Vector stores are
# cached in memory so each configuration is embedded only once per run.
configuration_vector_stores = {}
configuration_clients = {}


In [87]:
# Cell objective: Build and cache the SEC chunking and Chroma index for a configuration.
def build_configuration_vector_store(config):
    """Build and cache the vector index required by one RAG configuration."""
    config_name = config["name"]
    if config_name in configuration_vector_stores:
        return configuration_vector_stores[config_name]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )
    config_sec_docs = splitter.split_documents(sec_raw_docs)

    for chunk_id, doc in enumerate(config_sec_docs):
        doc.metadata["chunk_id"] = chunk_id

    config_docs = csv_docs + config_sec_docs
    config_ids = [deterministic_document_id(doc) for doc in config_docs]

    client = chromadb.EphemeralClient()
    store = Chroma(
        client=client,
        collection_name=f"rag_tuning_{config_name}",
        embedding_function=embeddings,
    )
    store.add_documents(config_docs, ids=config_ids)

    configuration_clients[config_name] = client
    configuration_vector_stores[config_name] = store
    print(
        f"Built {config_name}: {len(csv_docs)} CSV documents + "
        f"{len(config_sec_docs)} SEC chunks"
    )
    return store


In [88]:
# Cell objective: Reuse the shared source-aware retriever with configuration settings.
def retrieve_for_configuration(question, config):
    """Retrieve evidence with the shared router and a configuration-specific index."""
    store = build_configuration_vector_store(config)
    return retrieve_context(
        question=question,
        k=config["k"],
        search_type=config["search_type"],
        use_routing=config["use_routing"],
        store=store,
    )


In [89]:
# Cell objective: Generate an answer with one complete RAG configuration.
def answer_with_configuration(question, config):
    """Answer one question with a complete retrieval and generation configuration."""
    retrieval_result = retrieve_for_configuration(question, config)
    docs = retrieval_result["documents"]
    formatted_context = format_retrieved_context(docs)

    rendered_prompt = selected_prompt_for_tuning.interpolate(
        question=question,
        context=formatted_context,
    )
    llm = ChatOpenAI(
        model="gpt-4.1-mini",
        temperature=config["temperature"],
        top_p=config["top_p"],
        max_tokens=config["max_tokens"],
    )
    response = llm.invoke([
        SystemMessage(content=FINANCIAL_SAFETY_POLICY),
        HumanMessage(content=rendered_prompt),
    ])

    return {
        "answer": response.content,
        "documents": docs,
        "formatted_context": formatted_context,
        "retrieval_context": [doc.page_content for doc in docs],
        "retrieved_sources": [
            format_source_metadata(doc.metadata or {}) for doc in docs
        ],
        "diagnostics": retrieval_result["diagnostics"],
    }


## 3.3 Evaluate one configuration on a benchmark set

In [90]:
# Cell objective: Define how one complete RAG configuration is evaluated.
# Objective: evaluate one complete RAG configuration on a benchmark. Retrieval
# is intentionally rerun because it is the variable being tuned in Section 3.
def evaluate_rag_configuration(config, benchmark_df):
    """Run and formally evaluate one complete RAG configuration."""
    answer_rows = []
    evaluation_rows = []

    for case_number, (_, row) in enumerate(benchmark_df.iterrows(), start=1):
        question = row["question"]
        expected_answer = row["response"]
        print(
            f"{config['name']}: case {case_number}/{len(benchmark_df)}"
        )

        result = answer_with_configuration(question, config)
        retrieval_checks = retrieval_quality(
            row,
            {
                "documents": result["documents"],
                "retrieved_sources": result["retrieved_sources"],
            },
        )

        test_case = LLMTestCase(
            input=question,
            actual_output=result["answer"],
            expected_output=expected_answer,
            retrieval_context=result["retrieval_context"],
        )

        answer_rows.append({
            "config_name": config["name"],
            "case_id": int(row["case_id"]),
            "question": question,
            "expected_output": expected_answer,
            "actual_output": result["answer"],
            "retrieved_sources": result["retrieved_sources"],
            "retrieved_count": len(result["documents"]),
            "retrieval_diagnostics": result["diagnostics"],
            **retrieval_checks,
        })

        # Context Relevance belongs here because each configuration can retrieve
        # different evidence. The four generation metrics evaluate the answer.
        for metric in rag_evaluation_metrics:
            evaluation_rows.append({
                "config_name": config["name"],
                "case_id": int(row["case_id"]),
                "question": question,
                "metric": metric.name,
                "coverage_status": retrieval_checks["coverage_status"],
                "source_match": retrieval_checks["source_match"],
                **measure_metric(metric, test_case),
            })

    return pd.DataFrame(evaluation_rows), pd.DataFrame(answer_rows)


## 3.4 Run all configurations on the test set

In [91]:
# Cell objective: Prepare shared containers for fair configuration evaluation.
# Objective: run every configuration on the same benchmark cases. After using
# this holdout to select a configuration, treat it as validation data rather
# than as an untouched final test set.
tuning_benchmark_df = evaluation_df.copy()
configuration_evaluation_frames = []
configuration_answer_frames = []


In [92]:
# Cell objective: Run all three configurations on the same tuning benchmark.
for config in rag_configurations:
    print(f"\nEvaluating RAG configuration: {config['name']}")
    config_evaluation_df, config_answers_df = evaluate_rag_configuration(
        config, tuning_benchmark_df
    )
    configuration_evaluation_frames.append(config_evaluation_df)
    configuration_answer_frames.append(config_answers_df)

all_config_evaluations_df = pd.concat(
    configuration_evaluation_frames, ignore_index=True
)
all_config_answers_df = pd.concat(
    configuration_answer_frames, ignore_index=True
)

print(
    f"Completed {len(rag_configurations)} configurations across "
    f"{len(tuning_benchmark_df)} benchmark cases."
)



Evaluating RAG configuration: Config_1_Precision_Focused
Config_1_Precision_Focused: case 1/8
Built Config_1_Precision_Focused: 1000 CSV documents + 692 SEC chunks
Config_1_Precision_Focused: case 2/8
Config_1_Precision_Focused: case 3/8
Config_1_Precision_Focused: case 4/8
Config_1_Precision_Focused: case 5/8
Config_1_Precision_Focused: case 6/8
Config_1_Precision_Focused: case 7/8
Config_1_Precision_Focused: case 8/8

Evaluating RAG configuration: Config_2_Balanced
Config_2_Balanced: case 1/8
Built Config_2_Balanced: 1000 CSV documents + 522 SEC chunks
Config_2_Balanced: case 2/8
Config_2_Balanced: case 3/8
Config_2_Balanced: case 4/8
Config_2_Balanced: case 5/8
Config_2_Balanced: case 6/8
Config_2_Balanced: case 7/8
Config_2_Balanced: case 8/8

Evaluating RAG configuration: Config_3_Context_Heavy
Config_3_Context_Heavy: case 1/8
Built Config_3_Context_Heavy: 1000 CSV documents + 431 SEC chunks
Config_3_Context_Heavy: case 2/8
Config_3_Context_Heavy: case 3/8
Config_3_Context_Heavy:

## 3.5 Compare configurations

In [93]:
# Cell objective: Aggregate metric-level results for each RAG configuration.
# Objective: compare retrieval and generation quality together because both
# change across RAG configurations. The weights express project priorities.
configuration_metric_summary_df = (
    all_config_evaluations_df.groupby(["config_name", "metric"])
    .agg(
        average_score=("score", "mean"),
        success_rate=("success", "mean"),
        evaluated_cases=("score", "count"),
        error_count=("error", lambda values: values.notna().sum()),
    )
    .reset_index()
)

configuration_metric_scores_df = configuration_metric_summary_df.pivot(
    index="config_name",
    columns="metric",
    values="average_score",
).reset_index()


In [94]:
# Cell objective: Aggregate execution and deterministic retrieval-coverage results.
configuration_execution_summary_df = (
    all_config_evaluations_df.groupby("config_name")
    .agg(
        overall_average_score=("score", "mean"),
        overall_success_rate=("success", "mean"),
        error_count=("error", lambda values: values.notna().sum()),
    )
    .reset_index()
)

configuration_retrieval_summary_df = (
    all_config_answers_df.groupby("config_name")
    .agg(
        source_match_rate=("source_match", "mean"),
        supported_rate=(
            "coverage_status",
            lambda values: values.eq("SUPPORTED").mean(),
        ),
        average_retrieved_documents=("retrieved_count", "mean"),
    )
    .reset_index()
)


In [95]:
# Cell objective: Merge configuration parameters and evaluation summaries into one scorecard.
configuration_scorecard_df = (
    configuration_metric_scores_df
    .merge(configuration_execution_summary_df, on="config_name")
    .merge(configuration_retrieval_summary_df, on="config_name")
    .merge(
        configuration_table_df.rename(columns={"name": "config_name"}),
        on="config_name",
    )
)


In [96]:
# Cell objective: Apply project-priority weights and rank the configurations.
RAG_TUNING_WEIGHTS = {
    "Answer Correctness": 0.35,
    "Faithfulness / Groundedness": 0.25,
    "Context Relevance": 0.20,
    "Answer Relevance": 0.05,
    "Broker Actionability": 0.15,
}

missing_metrics = set(RAG_TUNING_WEIGHTS).difference(
    configuration_scorecard_df.columns
)
if missing_metrics:
    raise ValueError(f"Missing tuning metrics: {sorted(missing_metrics)}")

configuration_scorecard_df["weighted_score"] = sum(
    configuration_scorecard_df[metric] * weight
    for metric, weight in RAG_TUNING_WEIGHTS.items()
)

configuration_scorecard_df = configuration_scorecard_df.sort_values(
    ["weighted_score", "source_match_rate"],
    ascending=False,
).reset_index(drop=True)

print("RAG configuration scorecard")
display(configuration_scorecard_df.round(4))


RAG configuration scorecard


,config_name,Answer Correctness,Answer Relevance,Broker Actionability,Context Relevance,Faithfulness / Groundedness,overall_average_score,overall_success_rate,error_count,source_match_rate,...,chunk_size,chunk_overlap,k,search_type,use_routing,temperature,top_p,max_tokens,rationale,weighted_score
0,Config_1_Precision_Focused,0.7037,0.8644,0.8817,0.4604,0.9027,0.7626,0.825,0,1.0,...,700,100,4,similarity,True,0.0,1.0,450,Tests whether narrower retrieval reduces noise for focused questions.,0.7395
1,Config_3_Context_Heavy,0.7476,0.8685,0.8944,0.4199,0.8575,0.7576,0.850,0,1.0,...,1200,250,7,mmr,True,0.1,0.9,650,Tests broader and more diverse evidence for multi-part questions.,0.7376
2,Config_2_Balanced,0.6890,0.8773,0.8976,0.4559,0.8891,0.7618,0.825,0,1.0,...,1000,200,5,similarity,True,0.0,1.0,500,"Reference balance between retrieval recall, noise, and response length.",0.7331


## 3.6 Select the best overall approach

In [97]:
# Cell objective: Filter configurations through minimum quality gates.
# Objective: select the strongest configuration without allowing a high
# weighted average to hide weak correctness, grounding, or execution errors.
eligible_configurations_df = configuration_scorecard_df[
    (configuration_scorecard_df["Answer Correctness"] >= EVALUATION_THRESHOLD)
    & (
        configuration_scorecard_df["Faithfulness / Groundedness"]
        >= EVALUATION_THRESHOLD
    )
    & (configuration_scorecard_df["error_count"] == 0)
]


In [98]:
# Cell objective: Select and describe the strongest eligible RAG approach.
if eligible_configurations_df.empty:
    selection_pool_df = configuration_scorecard_df
    rag_selection_decision = "SELECTED_WITH_WARNINGS"
else:
    selection_pool_df = eligible_configurations_df
    rag_selection_decision = "SELECTED"

best_configuration_row = selection_pool_df.sort_values(
    ["weighted_score", "source_match_rate"],
    ascending=False,
).iloc[0]

best_rag_config_name = best_configuration_row["config_name"]
best_rag_config = next(
    config
    for config in rag_configurations
    if config["name"] == best_rag_config_name
)

best_overall_approach = {
    "prompt": selected_prompt_name,
    "rag_configuration": best_rag_config,
    "weighted_score": float(best_configuration_row["weighted_score"]),
    "decision": rag_selection_decision,
}

print(f"Best RAG configuration: {best_rag_config_name}")
print(f"Selection decision: {rag_selection_decision}")
print(f"Weighted score: {best_overall_approach['weighted_score']:.4f}")
display(best_configuration_row.to_frame(name="value"))


Best RAG configuration: Config_1_Precision_Focused
Selection decision: SELECTED
Weighted score: 0.7395


,value
config_name,Config_1_Precision_Focused
Answer Correctness,0.70371
Answer Relevance,0.864399
Broker Actionability,0.881684
Context Relevance,0.460388
Faithfulness / Groundedness,0.902732
overall_average_score,0.762582
overall_success_rate,0.825
error_count,0
source_match_rate,1.0


## 3.7 Save the tuning results


In [99]:
# Cell objective: Define derived-artifact paths without touching raw data.
# Objective: persist derived tuning artifacts outside data/raw so the
# experiment can be reviewed without changing the source datasets.
tuning_output_dir = Path("../data/eval")
if not tuning_output_dir.exists():
    tuning_output_dir = Path("data/eval")
tuning_output_dir.mkdir(parents=True, exist_ok=True)

evaluation_path = tuning_output_dir / "rag_tuning_evaluations.csv"
answers_path = tuning_output_dir / "rag_tuning_answers.csv"
metric_summary_path = tuning_output_dir / "rag_tuning_metric_summary.csv"
scorecard_path = tuning_output_dir / "rag_tuning_scorecard.csv"
selection_path = tuning_output_dir / "best_rag_configuration.json"


In [100]:
# Cell objective: Save tuning details, summaries, scorecard, and final selection.
all_config_evaluations_df.to_csv(evaluation_path, index=False)
all_config_answers_df.to_csv(answers_path, index=False)
configuration_metric_summary_df.to_csv(metric_summary_path, index=False)
configuration_scorecard_df.to_csv(scorecard_path, index=False)

selection_payload = {
    **best_overall_approach,
    "prompt_text": selected_prompt_for_tuning.text_template,
    "metric_weights": RAG_TUNING_WEIGHTS,
    "benchmark_case_ids": [
        int(case_id) for case_id in tuning_benchmark_df["case_id"].tolist()
    ],
}
with selection_path.open("w", encoding="utf-8") as file:
    json.dump(selection_payload, file, indent=2)

print("Saved RAG tuning artifacts:")
for path in [
    evaluation_path,
    answers_path,
    metric_summary_path,
    scorecard_path,
    selection_path,
]:
    print(f"- {path.resolve()}")


Saved RAG tuning artifacts:
- /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/data/eval/rag_tuning_evaluations.csv
- /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/data/eval/rag_tuning_answers.csv
- /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/data/eval/rag_tuning_metric_summary.csv
- /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/data/eval/rag_tuning_scorecard.csv
- /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/data/eval/best_rag_configuration.json


# Section 4: Final RAG Inference Pipeline with the Optimized Prompt and Best RAG Configuration

## 4.1 Define the final inference helper

In [101]:
# Cell objective: Verify that every selected pipeline component is available.
# Objective: expose one final inference function that uses the prompt approved
# in Section 2 and the best RAG configuration selected in Section 3.
required_final_objects = [
    "best_rag_config",
    "selected_prompt_for_tuning",
    "answer_with_configuration",
    "CORPUS_AS_OF_DATE",
    "FINANCIAL_SAFETY_POLICY",
]
missing_final_objects = [
    name for name in required_final_objects if name not in globals()
]
if missing_final_objects:
    raise RuntimeError(
        "Run Sections 1 through 3 before Section 4. Missing: "
        f"{missing_final_objects}"
    )


In [102]:
# Cell objective: Expose one final inference helper using the approved pipeline.
def final_rag_answer(question):
    """Answer with the approved prompt and selected RAG configuration."""
    result = answer_with_configuration(question, best_rag_config)
    return {
        "question": question,
        "answer": result["answer"],
        "source_documents": result["documents"],
        "sources": result["retrieved_sources"],
        "retrieval_diagnostics": result["diagnostics"],
        "config_name": best_rag_config["name"],
        "prompt_name": selected_prompt_name,
        "selection_decision": rag_selection_decision,
        "corpus_as_of_date": CORPUS_AS_OF_DATE,
    }

print(
    f"Final pipeline ready: prompt={selected_prompt_name}, "
    f"configuration={best_rag_config['name']}, "
    f"corpus_as_of={CORPUS_AS_OF_DATE}"
)


Final pipeline ready: prompt=optimized, configuration=Config_1_Precision_Focused, corpus_as_of=2026-04-23


## 4.2 Define the final test cases

In [103]:
# Cell objective: Define the five official project test questions and diagnostic expectations.
# Objective: preserve the five official project test questions verbatim.
# Additional fields support deterministic retrieval diagnostics in Section 4.3;
# they do not alter the questions or act as reference answers.
final_test_cases = [
    {
        "id": 1,
        "question": (
            "Compare the total revenue and net income reported by Apple, Amazon, "
            "and Alphabet in their latest 10-Q filings. Which company grew fastest "
            "year-over-year?"
        ),
        "expected_datasets": ["sec_filings"],
        "required_entities": ["Apple", "Amazon", "Alphabet"],
        "evaluation_focus": "multi-company filing coverage and abstention",
    },
    {
        "id": 2,
        "question": (
            "Based on this week's news sentiment around Amazon, would you classify "
            "the current signal as bullish or bearish, and does the price data support that?"
        ),
        "expected_datasets": ["global_news", "stock_price_details"],
        "required_entities": ["Amazon"],
        "evaluation_focus": "cross-source sentiment and price grounding",
    },
    {
        "id": 3,
        "question": (
            "Should I invest in Apple right now? I heard Tim Cook is stepping down — "
            "is that a red flag? How's the stock been doing, and is there anything I should "
            "be worried about with the company's financials or risks?"
        ),
        "expected_datasets": [
            "global_news", "stock_price_details", "sec_filings"
        ],
        "required_entities": ["Apple", "Tim Cook"],
        "evaluation_focus": "rumor verification, grounding, and safe framing",
    },
    {
        "id": 4,
        "question": (
            "Is now a good time to buy Bitcoin? News is saying it just crossed $78k — "
            "am I too late? What's the price trend looked like over the last few months, "
            "and what's driving the current rally?"
        ),
        "expected_datasets": ["global_news", "stock_price_details"],
        "required_entities": ["Bitcoin"],
        "evaluation_focus": "claim verification and temporal evidence",
    },
    {
        "id": 5,
        "question": (
            "I want to invest in AI. Out of Google, Amazon, and Apple, which one looks "
            "like the best bet right now? How are they performing, what are they saying "
            "about their AI business, and what's the buzz around them?"
        ),
        "expected_datasets": [
            "global_news", "stock_price_details", "sec_filings"
        ],
        "required_entities": ["Alphabet", "Amazon", "Apple"],
        "evaluation_focus": "multi-source comparison without unsupported advice",
    },
]


In [104]:
# Cell objective: Validate and display the official test-case specification.
if len(final_test_cases) != 5:
    raise ValueError("The project rubric requires exactly five final test cases.")

display(pd.DataFrame(final_test_cases)[
    ["id", "expected_datasets", "required_entities", "evaluation_focus"]
])


,id,expected_datasets,required_entities,evaluation_focus
0,1,[sec_filings],"[Apple, Amazon, Alphabet]",multi-company filing coverage and abstention
1,2,"[global_news, stock_price_details]",[Amazon],cross-source sentiment and price grounding
2,3,"[global_news, stock_price_details, sec_filings]","[Apple, Tim Cook]","rumor verification, grounding, and safe framing"
3,4,"[global_news, stock_price_details]",[Bitcoin],claim verification and temporal evidence
4,5,"[global_news, stock_price_details, sec_filings]","[Alphabet, Amazon, Apple]",multi-source comparison without unsupported advice


## 4.3 Run the final test cases

In [105]:
# Cell objective: Define reusable official-case source and entity coverage checks.
def official_case_coverage(test_case, docs):
    """Check expected dataset and entity coverage for an official test case."""
    retrieved_datasets = sorted({
        doc.metadata.get("dataset", "unknown") for doc in docs
    })
    missing_datasets = sorted(
        set(test_case["expected_datasets"]) - set(retrieved_datasets)
    )
    searchable_text = " ".join(
        [doc.page_content for doc in docs]
        + [
            " ".join(str(value) for value in (doc.metadata or {}).values())
            for doc in docs
        ]
    ).lower()
    missing_entities = [
        entity for entity in test_case["required_entities"]
        if entity.lower() not in searchable_text
    ]

    if not docs:
        status = "UNSUPPORTED"
    elif missing_datasets or missing_entities:
        status = "PARTIAL"
    else:
        status = "SUPPORTED"

    return {
        "retrieved_datasets": retrieved_datasets,
        "missing_datasets": missing_datasets,
        "missing_entities": missing_entities,
        "coverage_status": status,
    }


In [106]:
# Cell objective: Declare the baseline and three configured pipelines to compare.
official_pipeline_specs = [
    {
        "pipeline": "baseline_prompt__baseline_retriever",
        "pipeline_type": "baseline",
        "prompt_name": "baseline",
        "config_name": "baseline_retriever",
        "config": None,
    },
    *[
        {
            "pipeline": f"approved_prompt__{config['name']}",
            "pipeline_type": "configuration",
            "prompt_name": selected_prompt_name,
            "config_name": config["name"],
            "config": config,
        }
        for config in rag_configurations
    ],
]


In [107]:
# Cell objective: Define one reusable runner for all five official cases.
def run_official_pipeline(pipeline_spec, test_cases):
    """Run all five official cases through one pipeline specification."""
    rows = []
    for test_case in test_cases:
        try:
            if pipeline_spec["pipeline_type"] == "baseline":
                result = baseline_rag_answer(test_case["question"])
                docs = result["source_documents"]
                answer = result["answer"]
                sources = [format_source_metadata(doc.metadata or {}) for doc in docs]
                diagnostics = result["retrieval_diagnostics"]
            else:
                result = answer_with_configuration(test_case["question"], pipeline_spec["config"])
                docs = result["documents"]
                answer = result["answer"]
                sources = result["retrieved_sources"]
                diagnostics = result["diagnostics"]

            rows.append({
                "id": test_case["id"],
                "question": test_case["question"],
                "evaluation_focus": test_case["evaluation_focus"],
                "answer": answer,
                "sources": sources,
                "source_documents": docs,
                "retrieval_context": [doc.page_content for doc in docs],
                **official_case_coverage(test_case, docs),
                "pipeline": pipeline_spec["pipeline"],
                "pipeline_type": pipeline_spec["pipeline_type"],
                "prompt_name": pipeline_spec["prompt_name"],
                "config_name": pipeline_spec["config_name"],
                "retrieval_diagnostics": diagnostics,
                "error": None,
            })
        except Exception as exc:
            rows.append({
                "id": test_case["id"],
                "question": test_case["question"],
                "evaluation_focus": test_case["evaluation_focus"],
                "answer": None,
                "sources": [],
                "source_documents": [],
                "retrieval_context": [],
                "retrieved_datasets": [],
                "missing_datasets": test_case["expected_datasets"],
                "missing_entities": test_case["required_entities"],
                "coverage_status": "ERROR",
                "pipeline": pipeline_spec["pipeline"],
                "pipeline_type": pipeline_spec["pipeline_type"],
                "prompt_name": pipeline_spec["prompt_name"],
                "config_name": pipeline_spec["config_name"],
                "retrieval_diagnostics": None,
                "error": str(exc),
            })
    return rows


In [108]:
# Cell objective: Execute the complete 5-by-4 official test matrix once.
official_answer_rows = []
for pipeline_spec in official_pipeline_specs:
    print(f"Running official cases: {pipeline_spec['pipeline']}")
    official_answer_rows.extend(run_official_pipeline(pipeline_spec, final_test_cases))

official_test_answers_df = pd.DataFrame(official_answer_rows)
expected_answer_runs = len(final_test_cases) * len(official_pipeline_specs)
if len(official_test_answers_df) != expected_answer_runs:
    raise ValueError(
        f"Expected {expected_answer_runs} official answers; received {len(official_test_answers_df)}."
    )


Running official cases: baseline_prompt__baseline_retriever
Running official cases: approved_prompt__Config_1_Precision_Focused
Running official cases: approved_prompt__Config_2_Balanced
Running official cases: approved_prompt__Config_3_Context_Heavy


In [109]:
# Cell objective: Display the five answers produced by the selected final pipeline.
selected_pipeline_name = f"approved_prompt__{best_rag_config['name']}"
final_results_df = official_test_answers_df[
    official_test_answers_df["pipeline"] == selected_pipeline_name
].copy()

print("Selected-pipeline answers for the five official cases")
display(final_results_df[[
    "id", "question", "answer", "sources", "coverage_status",
    "missing_datasets", "missing_entities", "error",
]])


Selected-pipeline answers for the five official cases


,id,question,answer,sources,coverage_status,missing_datasets,missing_entities,error
5,1,"Compare the total revenue and net income reported by Apple, Amazon, and Alphabet in their latest 10-Q filings. Which company grew fastest year-over-year?","Based on the latest 10-Q filings:\n\n- Apple reported net sales of $92.96 billion for Q2 2026 (Source 2, p.12) and net income of $29.58 billion for the three months ended March 28, 2026, up from $24.78 billion in the same quarter of 2025 (Source 3, p.65). This represents a year-over-year net inc...","[sec_filings_10q.pdf | page 83, sec_filings_10q.pdf | page 75, sec_filings_10q.pdf | page 65, sec_filings_10q.pdf | page 78]",PARTIAL,[],"[Amazon, Alphabet]",None
6,2,"Based on this week's news sentiment around Amazon, would you classify the current signal as bullish or bearish, and does the price data support that?","The current news sentiment around Amazon is bullish. This is supported by the recent double upgrade from Bank of America and KeyBanc ahead of Amazon's Q1 earnings report, with price targets raised to $298 and $325 respectively, reflecting strong confidence in AWS growth accelerating to 28-30% (S...","[stock_price_details.csv | row 396, stock_price_details.csv | row 478, global_news.csv | Amazon Gets a Double Upgrade From BofA and KeyBanc Ahead of Earnings: Is $325 the New Floor? | 2026-04-20T15:30:48Z | row 489, global_news.csv | 4 Signs XRP Is Moving From Bearish to Bullish: Analyst | 2026-...",SUPPORTED,[],[],None
7,3,"Should I invest in Apple right now? I heard Tim Cook is stepping down — is that a red flag? How's the stock been doing, and is there anything I should be worried about with the company's financials or risks?","Apple's CEO Tim Cook is stepping down, but this transition is viewed as a sign of strength rather than uncertainty for the company (Source 5). The market reaction was a slight after-hours dip, indicating no major alarm among investors. Over Tim Cook's tenure, Apple shares gained 1,881%, a strong...","[stock_price_details.csv | row 338, stock_price_details.csv | row 406, sec_filings_10q.pdf | page 76, sec_filings_10q.pdf | page 63, global_news.csv | Apple’s CEO Transition Signals Strength, Not Uncertainty | 2026-04-21T18:56:51Z | row 268, global_news.csv | 36 S&P 500 Stocks Blew Away Tim Cook...",SUPPORTED,[],[],None
8,4,"Is now a good time to buy Bitcoin? News is saying it just crossed $78k — am I too late? What's the price trend looked like over the last few months, and what's driving the current rally?","Bitcoin's recent price action shows it trading around $76,000 to $78,000 as of late April 2026, with a notable rebound from about $71,200 in mid-March (Source 1, 3, 4). Over the last few months, Bitcoin's price has fluctuated significantly: it was above $78,600 at the end of January but dropped ...","[stock_price_details.csv | row 30, stock_price_details.csv | row 280, global_news.csv | Bitcoin Entering Final Bull Trap Below $80k Before a Crash to $52k | 2026-04-21T19:39:50Z | row 42, global_news.csv | Bitcoin Rebounds Strongly — Can Bulls Drive Price Toward $79,000 | 2026-04-20T20:30:55Z | ...",SUPPORTED,[],[],None
9,5,"I want to invest in AI. Out of Google, Amazon, and Apple, which one looks like the best bet right now? How are they performing, what are they saying about their AI business, and what's the buzz around them?","Based on the available information up to early 2026, here is a comparative overview of Google (Alphabet), Amazon, and Apple regarding their AI investment prospects and recent performance:\n\n1. **Stock Performance Snapshot (Late 2025):** \n - Alphabet (GOOGL) closed at approximately $283.34 o...","[global_news.csv | Software makers' best may not be good enough as AI fears mount | 2026-04-22T10:27:30Z | row 212, global_news.csv | AI companies raise $242B in Q1 2026, OpenAI leads with $122B round | 2026-04-21T07:34:45Z | row 116, stock_price_details.csv | row 314, stock_price_details.csv | ...",SUPPORTED,[],[],None


In [110]:
# Cell objective: Define reference-free formal evaluation for official answers.
def evaluate_official_answers(answer_df, metrics):
    """Evaluate reference-free official answers with applicable RAG metrics."""
    rows = []
    for _, answer_row in answer_df.iterrows():
        test_case = LLMTestCase(
            input=answer_row["question"],
            actual_output=answer_row["answer"] or "",
            retrieval_context=answer_row["retrieval_context"],
        )
        for metric in metrics:
            measurement = (
                {
                    "score": None,
                    "success": False,
                    "reason": None,
                    "error": answer_row["error"],
                }
                if answer_row["error"] is not None
                else measure_metric(metric, test_case)
            )
            rows.append({
                "pipeline": answer_row["pipeline"],
                "pipeline_type": answer_row["pipeline_type"],
                "prompt_name": answer_row["prompt_name"],
                "config_name": answer_row["config_name"],
                "case_id": int(answer_row["id"]),
                "metric": metric.name,
                "coverage_status": answer_row["coverage_status"],
                **measurement,
            })
    return pd.DataFrame(rows)


In [111]:
# Cell objective: Evaluate all official answers and verify the expected metric count.
official_test_evaluation_df = evaluate_official_answers(
    official_test_answers_df,
    official_test_evaluation_metrics,
)
expected_metric_runs = expected_answer_runs * len(official_test_evaluation_metrics)
if len(official_test_evaluation_df) != expected_metric_runs:
    raise ValueError(
        f"Expected {expected_metric_runs} metric results; received {len(official_test_evaluation_df)}."
    )


In [112]:
# Cell objective: Build and display compact coverage and metric scorecards.
official_metric_summary_df = (
    official_test_evaluation_df.groupby(["pipeline", "metric"])
    .agg(
        average_score=("score", "mean"),
        success_rate=("success", "mean"),
        evaluated_cases=("score", "count"),
        error_count=("error", lambda values: values.notna().sum()),
    )
    .reset_index()
)
official_scorecard_df = official_metric_summary_df.pivot(
    index="pipeline", columns="metric", values="average_score"
).reset_index()
official_coverage_summary_df = (
    official_test_answers_df.groupby(["pipeline", "coverage_status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("Official five-case coverage")
display(official_coverage_summary_df)
print("Official five-case metric summary")
display(official_metric_summary_df.round(4))
print("Official five-case scorecard")
display(official_scorecard_df.round(4))


Official five-case coverage


coverage_status,pipeline,PARTIAL,SUPPORTED
0,approved_prompt__Config_1_Precision_Focused,1,4
1,approved_prompt__Config_2_Balanced,1,4
2,approved_prompt__Config_3_Context_Heavy,0,5
3,baseline_prompt__baseline_retriever,1,4


Official five-case metric summary


,pipeline,metric,average_score,success_rate,evaluated_cases,error_count
0,approved_prompt__Config_1_Precision_Focused,Answer Relevance,0.8524,1.0,5,0
1,approved_prompt__Config_1_Precision_Focused,Broker Actionability,0.8598,1.0,5,0
2,approved_prompt__Config_1_Precision_Focused,Context Relevance,0.5140,0.6,5,0
3,approved_prompt__Config_1_Precision_Focused,Faithfulness / Groundedness,0.9161,1.0,5,0
4,approved_prompt__Config_2_Balanced,Answer Relevance,0.8957,1.0,5,0
5,approved_prompt__Config_2_Balanced,Broker Actionability,0.8772,1.0,5,0
6,approved_prompt__Config_2_Balanced,Context Relevance,0.5253,0.6,5,0
7,approved_prompt__Config_2_Balanced,Faithfulness / Groundedness,0.9083,1.0,5,0
8,approved_prompt__Config_3_Context_Heavy,Answer Relevance,0.9068,1.0,5,0
9,approved_prompt__Config_3_Context_Heavy,Broker Actionability,0.8953,1.0,5,0


Official five-case scorecard


metric,pipeline,Answer Relevance,Broker Actionability,Context Relevance,Faithfulness / Groundedness
0,approved_prompt__Config_1_Precision_Focused,0.8524,0.8598,0.5140,0.9161
1,approved_prompt__Config_2_Balanced,0.8957,0.8772,0.5253,0.9083
2,approved_prompt__Config_3_Context_Heavy,0.9068,0.8953,0.5198,0.9060
3,baseline_prompt__baseline_retriever,0.8658,0.8819,0.5274,0.9115


In [113]:
# Cell objective: Save derived official-test artifacts outside the raw-data directory.
official_output_dir = Path("../data/eval") if Path("../data").exists() else Path("data/eval")
official_output_dir.mkdir(parents=True, exist_ok=True)
official_test_answers_df.drop(
    columns=["source_documents", "retrieval_context"]
).to_csv(official_output_dir / "official_test_answers.csv", index=False)
official_test_evaluation_df.to_csv(
    official_output_dir / "official_test_evaluations.csv", index=False
)
official_scorecard_df.to_csv(
    official_output_dir / "official_test_scorecard.csv", index=False
)
print(f"Saved official test artifacts to: {official_output_dir.resolve()}")


Saved official test artifacts to: /Users/mbagnara/Desktop/UofTexas/grounded-investment-assistant/data/eval


# Conclusion

## Actionable Insights:

The project produced four business-relevant insights from the implemented RAG pipeline and the five official test cases.

### 1. The assistant can reduce information overload

The system brings together information that brokers would otherwise need to review across separate sources:

- financial news;
- market-price records; and
- regulatory filings.

This unified research workflow can reduce manual searching and help brokers identify relevant market signals more efficiently.

> **Aligned business objectives:** Reduce information overload and improve market coverage.

### 2. The assistant can accelerate response preparation

The assistant retrieves evidence, combines information from multiple sources, and prepares an initial client-ready response with source references. Brokers can therefore spend less time collecting information and more time validating it, interpreting its implications, and adapting the response to the client's needs.

Across the official evaluation, every pipeline completed the five business questions without execution errors and achieved a 90% metric success rate. This provides encouraging proof-of-concept evidence that the workflow can support timely broker research.

> **Aligned business objective:** Respond more quickly to client questions.

### 3. The assistant can improve answer quality and traceability

The generated responses connect conclusions to retrieved evidence and identify the sources used. When the corpus did not contain the Amazon and Alphabet filings required for the multi-company comparison, the assistant disclosed the missing information instead of inventing figures or selecting an unsupported winner.

This transparent behavior can help brokers distinguish between supported conclusions, partial evidence, and questions that require additional research before communicating with a client.

> **Aligned business objectives:** Produce evidence-backed responses and reduce the risk of incomplete or fabricated information.

### 4. The assistant should augment, not replace, broker judgment

The project shows that the system can summarize financial signals, risks, price movements, news, and filing disclosures, but response quality remains dependent on the evidence available in the corpus. The assistant is therefore most valuable as a decision-support and research tool rather than as an autonomous investment adviser.

Brokers should remain responsible for reviewing the evidence, considering client suitability, resolving missing information, and making the final investment judgment.

> **Aligned business objective:** Support brokers without replacing their professional judgment.

## Recommendations:

The project findings support six business-oriented recommendations for moving from an educational proof of concept toward a controlled broker-support workflow.

### 1. Launch a controlled pilot with brokers

The organization should test the assistant with a small group of brokers for clearly defined activities such as:

- preparing for client meetings;
- producing market briefings;
- researching financial questions; and
- locating supporting evidence and sources.

The assistant should prepare research and draft responses, while brokers remain responsible for validating and communicating the final conclusion.

> **Aligned business objectives:** Respond more quickly to client questions and support brokers without replacing their professional judgment.

### 2. Display sources, the corpus cutoff date, and missing evidence

Every answer should clearly identify the supporting sources, the date through which the available corpus is current, and any information that could not be retrieved. The interface should also communicate whether evidence coverage is complete, partial, or insufficient.

This information will help brokers decide whether a response is ready for use or requires additional research before a client conversation.

> **Aligned business objectives:** Produce evidence-backed responses and reduce the risk of incomplete or fabricated information.

### 3. Select the retrieval strategy according to the question

The evaluation showed that one configuration is not optimal for every business question. The organization should use:

- focused retrieval for specific factual questions;
- broader and more diverse retrieval for multi-company comparisons; and
- multi-source retrieval for questions that combine news, prices, and regulatory risks.

A question-aware strategy can improve evidence coverage without adding unnecessary context to every response.

> **Aligned business objectives:** Improve market coverage and produce higher-quality responses.

### 4. Require human review for high-impact questions

Responses involving personalized investment decisions, buy or sell requests, executive-transition rumors, regulatory concerns, or client suitability should require broker review. The assistant should present evidence, alternative interpretations, and risks without making an autonomous investment decision.

> **Aligned business objectives:** Reduce information risk and preserve professional broker judgment.

### 5. Measure business value in addition to technical quality

A pilot should measure whether the assistant improves the broker's daily workflow through indicators such as:

- research time saved per question;
- percentage of responses accepted by brokers;
- number of corrections required;
- citation accuracy;
- frequency of appropriate abstention; and
- broker satisfaction.

These measures will show whether improvements in RAG metrics translate into practical operational value.

> **Aligned business objectives:** Reduce information overload, accelerate response preparation, and improve service quality.

### 6. Expand evidence coverage before production deployment

The official multi-company test could not be fully completed because the supplied corpus contained Apple's filing but not the corresponding Amazon and Alphabet filings. A production implementation should ensure that the expected company, reporting period, and source coverage are available for the questions the business intends to support.

The educational project should keep its supplied raw data unchanged and document this case as a known corpus limitation.

> **Aligned business objectives:** Improve market coverage and reduce the risk of presenting incomplete comparisons.

<font size=6>Power Ahead!</font>
___